# DMRS generation — TDL/CDL + scoped pure-Python COST2100/QuaDRiGa ports

No MATLAB/Octave runtime. Native channel generation, physical FIR application and RS estimation are connected.
The native ports are under validation and do not cover the full upstream API. Unsupported scenario features fail explicitly.
Do not interpret source-equation ports as validated MATLAB numerical parity.


In [ ]:
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass, field, asdict
from contextlib import ExitStack
from itertools import product
from types import SimpleNamespace
import copy, hashlib, json, math, os, tempfile
import numpy as np
import torch
import h5py
import matplotlib.pyplot as plt
from tqdm import tqdm
import neoradium
from neoradium import AntennaPanel, BandwidthPart, CdlChannel, TdlChannel, Modem
from neoradium import PDSCH as NrPDSCH

TRAIN_SAMPLES = 100_000  # Fix: 환경별 수량 / Mix: 전체 Train 총수
VAL_SAMPLES = 30_000     # Fix: 환경별 수량 / Mix: 전체 Val 총수
TEST_SAMPLES = 20_000    # Fix: 환경당 Test / Mix: Test 목록 전체의 총수
# 생성 모드는 아래 모든 채널 모델에 공통으로 적용합니다.
DATA_MODE = 'Mix'  # 'Fix': 환경당 10만/3만/2만 / 'Mix': Train/Val 총 10만/3만 + Random Test 총 2만
CHANNEL_TYPE_LIST = ['CDL-C']  # 예: ['TDL-A','CDL-C'] 또는 ['TDL','CDL']

# 'TDL' / 'CDL' 전체 계열을 선택했을 때 전개할 프로파일입니다.
TDL_PROFILE_LIST = ['A','B','C','D','E']
CDL_PROFILE_LIST = ['A','B','C','D','E']

SNR_LIST_DB = [-10,-5,0,5,10,15,20]  # Fix: 환경별 Train/Val/Test / Mix: Random Test의 환경 목록
DELAY_SPREAD_LIST_NS = [25,75,150,200,300]
DOPPLER_LIST_HZ = [50,200,500]

# Mix Train/Val의 환경 목록. 채널 종류는 위 CHANNEL_TYPE_LIST를 공통으로 사용합니다.
MIX_SNR_LIST = [10,15,20]
MIX_DELAY_LIST_NS = [25,75,150,200,300]
MIX_DOPPLER_LIST_HZ = [50,200,500]
MIX_SEED = 42

# COST2100/QuaDRiGa: 순수 Python source-equation port + 실제 DMRS 파형 통과.
# 현재 범위: 1 BS/1 MS, COST omni, QuaDRiGa legacy/mmMAGIC PAS 단일 이동 트랙.
# GR/추가 TOA도 native 분기로 계산. 미이식 Laplacian/다중 트랙 등은 명시적으로 거부합니다.
# 외부 .m 실행기와 MATLAB/Octave는 사용하지 않습니다.
COST2100_SCENARIO_LIST = ['IndoorHall_5GHz']
COST2100_SETTINGS = {
    'IndoorHall_5GHz': {
        'scenario':'LOS', 'link':'Single', 'band':'Wideband',
        'frequency_range_hz':[5.29e9,5.31e9],
        'bs_position_m':[10,10,0], 'ms_position_m':[10,5,0],
        'ms_velocity_mps':[0,-0.1,0],
    },
    'SemiUrban_300MHz': {
        'scenario':'LOS', 'link':'Single', 'band':'Wideband',
        'frequency_range_hz':[2.75e8,2.95e8],
        'bs_position_m':[0,0,0], 'ms_position_m':[100,-200,0],
        'ms_velocity_mps':[-0.2,0.9,0],
    },
}
# 아래 위치는 공식 demo의 첫 MS/BS 예시. Indoor close-user는 demo처럼 좌표 중심을 이동했습니다.
COST2100_SETTINGS.update({
    'Indoor_CloselySpacedUser_2_6GHz': {
        'scenario':'LOS','link':'Single','band':'Wideband','frequency_range_hz':[2.58e9,2.62e9],
        'bs_position_m':[3.668888888888889,-7.385555555555555,0.456666666666667],
        'ms_position_m':[0.8088888888888888,-1.2855555555555553,-0.5133333333333332],
        'ms_velocity_mps':[-0.25,0,0],
    },
    'SemiUrban_CloselySpacedUser_2_6GHz': {
        'scenario':'LOS','link':'Single','band':'Wideband','frequency_range_hz':[2.58e9,2.62e9],
        'bs_position_m':[0,0,8],'ms_position_m':[-27,-10,0],'ms_velocity_mps':[0.4,0.3,0],
    },
    'SemiUrban_VLA_2_6GHz': {
        'scenario':'LOS','link':'Single','band':'Wideband','frequency_range_hz':[2.57e9,2.62e9],
        'bs_position_m':[0,0,0],'ms_position_m':[30,30,0],'ms_velocity_mps':[0,0,0],
        'bs_spacing_m':[0.0577,0,0],'bs_position_count':128,
    },
})

QUADRIGA_SCENARIO_LIST = ['3GPP_38.901_UMi_NLOS']
QUADRIGA_SETTINGS = {
    'frequency_hz':3.5e9,
    'tx_position_m':[0,0,10], 'rx_position_m':[100,0,1.5],
    'rx_velocity_mps':[1,0,0],
    'absolute_delays':True,  # native carrier phase와 양의 전파 지연을 함께 유지
    'power_calibration_samples':32,  # 고정 ensemble 정규화; 개별 샘플 정규화 아님
}

QUADRIGA_SCENARIO_SETTINGS = {}  # 시나리오별 공통 설정 덮어쓰기: {'3GPP_38.901_UMa_LOS': {'tx_position_m':[0,0,25]}}

SEED = 20261021

N_SLOTS = 1               # 한 샘플당 연속 슬롯 수. 1이면 Slot 0, 5이면 Slot 0-4
N_TX = 1  # 1~4: 물리 송신 안테나마다 DM-RS 포트 하나, identity mapping
N_RX = 1  # 수신 안테나 수. Tx=1이면 Rx가 여러 개여도 CDM Off
N_RBS = 10                 # Type-1 DM-RS: 6 RE/RB → 60 RS per symbol
SCS_KHZ = 15  # 이 실험은 15 kHz 고정; 저장 경로에는 포함하지 않음
DMRS_SYMBOLS = (2, 11)      # 두 심볼의 RS subcarrier 위치는 반드시 동일
INTERP = False              # False: RS 60×2 / True: bounded 2D linear 119×10
DATA_MODULATION = '16QAM'   # 데이터는 3…10번 심볼, RS의 첫…마지막 subcarrier에만 배치
DATA_ROOT = Path('E:/DMRS Channel')  # 이 아래 SISO / MISO (Nx1) / SIMO / MIMO 분기 자동 생성
COMPUTE_DEVICE = 'cuda'    # 'cuda', 'cpu', 'auto'
NORMALIZE_AVERAGE_POWER = True  # 고정 ensemble 전력 정규화. False면 native 경로손실 크기까지 그대로 저장
COMPUTE_DTYPE = 'complex64'
GENERATION_BATCH_SIZE = 1000  # 1슬롯 SISO 기준 배치; 슬롯 수와 Tx×Rx에 따라 메모리 기준 자동 축소
PATH_CHUNK_SIZE = 4
TDL_MIMO_CORRELATION = 'Low'  # NeoRadium: Low / Medium / MediumA / MediumB / High
TDL_NUM_SINUSOIDS = 32  # NeoRadium Xiao SOS. 한 샘플의 모든 슬롯은 같은 phase로 연결
HDF5_COMPRESSION = 'lzf'
RESUME = True
OVERWRITE = False
SHOW_PROGRESS = True

print('NeoRadium:', neoradium.__version__, '| Torch:', torch.__version__)

## 설정과 공통 도구


In [ ]:
# 공식 QuaDRiGa config 이름: 사용자 설정은 위 SCENARIO_LIST에서 선택합니다.
QUADRIGA_OFFICIAL_SCENARIOS = ('3GPP_37.885_Highway_LOS', '3GPP_37.885_Highway_NLOSv', '3GPP_37.885_Urban_LOS', '3GPP_37.885_Urban_NLOS', '3GPP_37.885_Urban_NLOSv', '3GPP_38.901_InF_LOS', '3GPP_38.901_InF_NLOS_DH', '3GPP_38.901_InF_NLOS_DL', '3GPP_38.901_InF_NLOS_SH', '3GPP_38.901_InF_NLOS_SL', '3GPP_38.901_Indoor_LOS', '3GPP_38.901_Indoor_NLOS', '3GPP_38.901_RMa_LOS', '3GPP_38.901_RMa_LOS_O2I', '3GPP_38.901_RMa_NLOS', '3GPP_38.901_RMa_NLOS_O2I', '3GPP_38.901_UMa_LOS', '3GPP_38.901_UMa_LOS_O2I', '3GPP_38.901_UMa_NLOS', '3GPP_38.901_UMa_NLOS_O2I', '3GPP_38.901_UMi_LOS', '3GPP_38.901_UMi_LOS_GR', '3GPP_38.901_UMi_LOS_O2I', '3GPP_38.901_UMi_NLOS', '3GPP_38.901_UMi_NLOS_O2I', '3GPP_3D_UMa_LOS', '3GPP_3D_UMa_LOS_O2I', '3GPP_3D_UMa_NLOS', '3GPP_3D_UMa_NLOS_O2I', '3GPP_3D_UMi_LOS', '3GPP_3D_UMi_LOS_O2I', '3GPP_3D_UMi_NLOS', '3GPP_3D_UMi_NLOS_O2I', '5G-ALLSTAR_DenseUrban_LOS', '5G-ALLSTAR_DenseUrban_NLOS', '5G-ALLSTAR_Rural_LOS', '5G-ALLSTAR_Rural_NLOS', '5G-ALLSTAR_Suburban_LOS', '5G-ALLSTAR_Suburban_NLOS', '5G-ALLSTAR_Urban_LOS', '5G-ALLSTAR_Urban_NLOS', 'BERLIN_UMa_LOS', 'BERLIN_UMa_NLOS', 'BERLIN_UMi_Campus_LOS', 'BERLIN_UMi_Campus_NLOS', 'BERLIN_UMi_Square_LOS', 'BERLIN_UMi_Square_NLOS', 'DRESDEN_UMa_LOS', 'DRESDEN_UMa_NLOS', 'Freespace', 'LOSonly', 'MIMOSA_10-45_LOS', 'MIMOSA_10-45_NLOS', 'MIMOSA_16-25_LOS', 'MIMOSA_16-25_NLOS', 'MIMOSA_25-35_LOS', 'MIMOSA_25-35_NLOS', 'MIMOSA_35-45_LOS', 'MIMOSA_35-45_NLOS', 'Null', 'QuaDRiGa_Industrial_LOS', 'QuaDRiGa_Industrial_NLOS', 'QuaDRiGa_NTN_DenseUrban_LOS', 'QuaDRiGa_NTN_DenseUrban_NLOS', 'QuaDRiGa_NTN_Rural_LOS', 'QuaDRiGa_NTN_Rural_NLOS', 'QuaDRiGa_NTN_Suburban_LOS', 'QuaDRiGa_NTN_Suburban_NLOS', 'QuaDRiGa_NTN_Urban_LOS', 'QuaDRiGa_NTN_Urban_NLOS', 'QuaDRiGa_UD2D_LOS', 'QuaDRiGa_UD2D_NLOS', 'TwoRayGR', 'WINNER_Indoor_A1_LOS', 'WINNER_Indoor_A1_NLOS', 'WINNER_SMa_C1_LOS', 'WINNER_SMa_C1_NLOS', 'WINNER_UMa2Indoor_C4_LOS', 'WINNER_UMa2Indoor_C4_NLOS', 'WINNER_UMa_C2_LOS', 'WINNER_UMa_C2_NLOS', 'WINNER_UMi2Indoor_B4_LOS', 'WINNER_UMi2Indoor_B4_NLOS', 'WINNER_UMi_B1_LOS', 'WINNER_UMi_B1_NLOS', 'mmMAGIC_Indoor_LOS', 'mmMAGIC_Indoor_NLOS', 'mmMAGIC_UMi_LOS', 'mmMAGIC_UMi_LOS_O2I', 'mmMAGIC_UMi_NLOS', 'mmMAGIC_UMi_NLOS_O2I')

def normalized_data_mode(value):
    if not isinstance(value,str) or value.lower() not in ('fix','mix'):
        raise ValueError("DATA_MODE는 'Fix' 또는 'Mix'여야 합니다.")
    return value.title()


def expanded_channel_types(values):
    if not values:raise ValueError('CHANNEL_TYPE_LIST가 비어 있습니다.')
    result=[]
    for value in values:
        if value in ('TDL','CDL'):
            profiles=TDL_PROFILE_LIST if value=='TDL' else CDL_PROFILE_LIST
            if not profiles:raise ValueError(value+' 프로파일 List가 비어 있습니다.')
            selected=[]
            for profile in profiles:
                if profile not in ('A','B','C','D','E'):raise ValueError(value+' 프로파일은 A~E입니다.')
                selected.append(value+'-'+profile)
        elif value in ('COST2100','QuaDRiGa','QUADRIGA'):
            selected=[value]
        elif value in [b+'-'+p for b in ('TDL','CDL') for p in 'ABCDE']:
            selected=[value]
        else:raise ValueError('지원하지 않는 채널 종류: '+str(value))
        for name in selected:
            if name in result:raise ValueError('선택한 채널/프로파일이 중복됩니다: '+name)
            result.append(name)
    return result


def _finite_vector(values,name):
    array=np.asarray(values,dtype=float)
    if array.shape!=(3,) or not np.isfinite(array).all():raise ValueError(name+'은 유한한 x,y,z 3개 값이어야 합니다.')
    return array


def validate_native_scenarios(selected):
    # 공식 Git에서 지원한다고 명시한 조합을 검사합니다.
    # 예제에 나온 주파수/위치 값은 기본값이며, 문서 없는 임의의 허용 범위를 만들지 않습니다.
    if 'COST2100' in selected:
        allowed={
            'IndoorHall_5GHz': ('LOS',),
            'SemiUrban_300MHz': ('LOS','NLOS'),
            'Indoor_CloselySpacedUser_2_6GHz': ('LOS',),
            'SemiUrban_CloselySpacedUser_2_6GHz': ('LOS','NLOS'),
            'SemiUrban_VLA_2_6GHz': ('LOS','NLOS'),
        }
        if not COST2100_SCENARIO_LIST:raise ValueError('COST2100_SCENARIO_LIST가 비어 있습니다.')
        if len(set(COST2100_SCENARIO_LIST))!=len(COST2100_SCENARIO_LIST):raise ValueError('COST2100 시나리오가 중복됩니다.')
        for scenario in COST2100_SCENARIO_LIST:
            if scenario not in allowed:raise ValueError('공식 COST2100에 없는 network: '+scenario)
            if scenario not in COST2100_SETTINGS:raise ValueError('COST2100 설정이 없습니다: '+scenario)
            settings=COST2100_SETTINGS[scenario];link=settings['link'];los=settings['scenario']
            if link not in ('Single','Multiple') or los not in allowed[scenario]:
                raise ValueError(f'COST2100 지원 조합이 아닙니다: {scenario}/{link}/{los}')
            if scenario=='IndoorHall_5GHz' and link!='Single':
                raise ValueError('공식 demo의 IndoorHall_5GHz는 Single link 조합만 지원합니다.')
            if settings['band'] not in ('Wideband','Narrowband'):raise ValueError('COST2100 band 설정이 잘못되었습니다.')
            frequency=np.asarray(settings['frequency_range_hz'],float)
            if frequency.shape!=(2,) or not np.isfinite(frequency).all() or not 0<frequency[0]<frequency[1]:
                raise ValueError('COST2100 frequency_range_hz는 양수의 [시작, 끝]이어야 합니다.')
            _finite_vector(settings['bs_position_m'],'BS 위치')
            _finite_vector(settings['ms_position_m'],'MS 위치')
            _finite_vector(settings['ms_velocity_mps'],'MS 속도')
    if any(v in selected for v in ('QuaDRiGa','QUADRIGA')):
        if not QUADRIGA_SCENARIO_LIST:raise ValueError('QUADRIGA_SCENARIO_LIST가 비어 있습니다.')
        if len(set(QUADRIGA_SCENARIO_LIST))!=len(QUADRIGA_SCENARIO_LIST):raise ValueError('QuaDRiGa 시나리오가 중복됩니다.')
        for scenario in QUADRIGA_SCENARIO_LIST:
            if scenario not in QUADRIGA_OFFICIAL_SCENARIOS:raise ValueError('공식 QuaDRiGa config에 없는 시나리오: '+scenario)
        settings=QUADRIGA_SETTINGS
        if not np.isfinite(settings['frequency_hz']) or settings['frequency_hz']<=0:raise ValueError('QuaDRiGa frequency_hz는 양수여야 합니다.')
        _finite_vector(settings['tx_position_m'],'Tx 위치')
        _finite_vector(settings['rx_position_m'],'Rx 위치')
        _finite_vector(settings['rx_velocity_mps'],'Rx 속도')


def validate_channel_selection(values):
    selected=expanded_channel_types(values)
    validate_native_scenarios(selected)
    return selected


@dataclass
class CarrierSettings:
    n_rbs: int
    scs_khz: int  # 맨 위 SCS_KHZ가 전달됩니다. 기본값을 중복 지정하지 않습니다.
    cp_type: str = "normal"
    cell_id: int = 1

@dataclass
class AntennaSettings:
    tx_shape: tuple[int, int] = (1, 1)
    rx_shape: tuple[int, int] = (1, 1)
    tx_polarization: str = "|"
    rx_polarization: str = "|"
    spacing_lambda: tuple[float, float] = (0.5, 0.5)
    tx_orientation_deg: tuple[float, float, float] = (0, 0, 0)
    rx_orientation_deg: tuple[float, float, float] = (180, 0, 0)
    tx_beam_width_deg: tuple[float, float] = (65, 65)
    rx_beam_width_deg: tuple[float, float] = (65, 360)

@dataclass
class ChannelSettings:
    carrier_frequency_hz: float = 3.5e9
    ue_direction_deg: tuple[float, float] = (0, 90)  # azimuth, zenith
    k_factor_db: float | None = None
    xpr_db: float | None = None
    angle_means_deg: tuple[float, float, float, float] | None = None
    angle_spreads_deg: tuple[float, float, float, float] | None = None
    normalize_gains: bool = True
    normalize_output: bool = False
    normalize_average_power: bool = False  # Notebook enables unit mean |H|^2, including antenna patterns
    power_calibration_samples: int = 256   # Independent geometry/phase draws; fixed scale per condition
    power_calibration_seed: int = 20261018
    tdl_mimo_correlation: str = 'Low'
    tdl_num_sinusoids: int = 32
    filter_len: int = 16
    delay_quant_size: int = 64    # Fractional-delay FIR resolution; NOT ADC quantization
    compensate_fir_latency: bool = True

@dataclass
class DmrsGenerationConfig:
    carrier: CarrierSettings
    antennas: AntennaSettings = field(default_factory=lambda: AntennaSettings(tx_shape=(1,1), rx_shape=(1,1)))
    channel: ChannelSettings = field(default_factory=lambda: ChannelSettings(normalize_average_power=True))
    split_counts: tuple[int,int,int] = (100_000,30_000,20_000)
    seed: int = 20261021
    n_slots: int = 1
    interpolate_2d: bool = True
    dmrs_symbols: tuple[int,int] = (2,11)
    modulation: str = '16QAM'
    noise_enabled: bool = True
    data_mode: str = 'Mix'
    estimator: str = 'per_re_siso_cdm_multiport_v1'
    tx_epre: float = 1.0
    compute_dtype: str = 'complex64'

    # Test-only storage keeps the original physical sample coordinates for replay.
    source_split_counts: tuple[int,int,int] | None = None
    source_selection_seed: int = 42

    @property
    def sample_count(self):
        return sum(int(n) for n in self.split_counts)

def make_notebook_settings():
    """맨 위 설정 셀의 현재 값을 읽습니다. 생성 직전에 다시 호출합니다."""
    if SCS_KHZ != 15:
        raise ValueError('이 DMRS 실험의 SCS는 15 kHz로 고정되어 있습니다.')
    config = DmrsGenerationConfig(
        carrier=CarrierSettings(n_rbs=N_RBS, scs_khz=SCS_KHZ),
        antennas=AntennaSettings(tx_shape=(1,N_TX),rx_shape=(1,N_RX)),
        channel=ChannelSettings(normalize_average_power=NORMALIZE_AVERAGE_POWER,tdl_mimo_correlation=TDL_MIMO_CORRELATION,tdl_num_sinusoids=TDL_NUM_SINUSOIDS),
        data_mode=normalized_data_mode(DATA_MODE),
        split_counts=(TRAIN_SAMPLES, VAL_SAMPLES, TEST_SAMPLES), seed=SEED, n_slots=N_SLOTS,
        interpolate_2d=INTERP, dmrs_symbols=tuple(DMRS_SYMBOLS),
        modulation=DATA_MODULATION, compute_dtype=COMPUTE_DTYPE)
    execution = SimpleNamespace(
        device=COMPUTE_DEVICE, batch_size=GENERATION_BATCH_SIZE,
        path_chunk_size=PATH_CHUNK_SIZE, compression=HDF5_COMPRESSION,
        resume=RESUME, overwrite=OVERWRITE, show_progress=SHOW_PROGRESS)
    return config, execution


CONFIG, EXECUTION = make_notebook_settings()

FORMAT = 'dmrs-single-slot-ce-3.0'
MULTI_SLOT_FORMAT = 'dmrs-multi-slot-ce-1.0'
_CDL_POWER_REFERENCES = {}

def _jsonable(x):
    if isinstance(x, complex):
        return {"real": x.real, "imag": x.imag}
    if isinstance(x, np.ndarray):
        return _jsonable(x.tolist())
    if isinstance(x, np.generic):
        return _jsonable(x.item())
    if isinstance(x, dict):
        return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (tuple, list)):
        return [_jsonable(v) for v in x]
    if isinstance(x, Path):
        return str(x)
    return x

def _canonical(x):
    return json.dumps(_jsonable(x), ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def _antenna_ports(shape, pol):
    return int(np.prod(shape)) * (2 if pol in ("x", "+") else 1)

def _bwp(cfg):
    # Active BWP subcarriers are centered in the baseband FFT. RB indices are BWP-relative.
    return BandwidthPart(startRb=0, numRbs=cfg.carrier.n_rbs, spacing=cfg.carrier.scs_khz,
                         cpType=cfg.carrier.cp_type.lower(), cellId=cfg.carrier.cell_id)

def _arrays(cfg):
    a = cfg.antennas
    tx = AntennaPanel(list(a.tx_shape), polarization=a.tx_polarization, spacing=list(a.spacing_lambda),
                      beamWidth=list(a.tx_beam_width_deg))
    rx = AntennaPanel(list(a.rx_shape), polarization=a.rx_polarization, spacing=list(a.spacing_lambda),
                      beamWidth=list(a.rx_beam_width_deg))
    return tx, rx


## NeoRadium CDL / TDL 채널 — 연속 시간, 분수 지연 FIR, 고정 앙상블 전력 정규화


In [ ]:
def _raw_cdl_channel(cfg, bwp, profile, ds, fd, seed):
    tx, rx = _arrays(cfg)
    c = cfg.channel
    kw = dict(delaySpread=ds, carrierFreq=c.carrier_frequency_hz, dopplerShift=fd,
              txAntenna=tx, rxAntenna=rx, txOrientation=list(cfg.antennas.tx_orientation_deg),
              rxOrientation=list(cfg.antennas.rx_orientation_deg), ueDirAZ=list(c.ue_direction_deg),
              normalizeGains=c.normalize_gains, normalizeOutput=c.normalize_output,
              filterLen=c.filter_len, delayQuantSize=c.delay_quant_size, seed=int(seed))
    if c.k_factor_db is not None:
        if profile not in ("D", "E"):
            raise ValueError("K-factor scaling is only applicable to LOS CDL-D/E.")
        kw["kFactor"] = c.k_factor_db
    if c.xpr_db is not None:
        kw["xPolPower"] = c.xpr_db
    if c.angle_means_deg is not None:
        kw["angleScaling"] = (np.asarray(c.angle_means_deg), np.asarray(c.angle_spreads_deg))
    return CdlChannel(bwp, profile=profile, **kw)

def _cdl_spatial_rays(ch):
    """Exact NeoRadium ray coefficients and Dopplers, before summation over rays."""
    ray_count = 20
    frequencies = []
    def select_rays(theta, phi):
        az, zen = np.deg2rad(ch.ueDirAZ)
        velocity = ch.dopplerShift*np.array([np.sin(zen)*np.cos(az), np.sin(zen)*np.sin(az), np.cos(zen)])
        t, p = np.deg2rad(theta), np.deg2rad(phi)
        direction = np.stack([np.sin(t)*np.cos(p), np.sin(t)*np.sin(p), np.cos(t)], axis=-1)
        frequency = (direction*velocity).sum(-1)
        if theta.shape[1] == 1:
            frequencies.append(np.repeat(frequency, ray_count, axis=1))
            return np.ones((*theta.shape, ray_count))
        frequencies.append(frequency)
        return np.broadcast_to(np.eye(ray_count), (len(theta), ray_count, ray_count))
    ch.getDopplerFactor = select_rays
    try:
        spatial = ch.getChannelGains()
    finally:
        del ch.getDopplerFactor
    if ch.hasLos:
        spatial[1:, :, :, 0] = 0  # One LOS ray, not twenty copies
    return spatial, np.concatenate(frequencies, axis=0)

def _cdl_power_reference(cfg, profile, ds):
    """Fixed ensemble reference, independent of data samples, SNR and Doppler.

    Average incoherent ray powers (including polarization/antenna patterns and
    active-band FIR response). Random phase interference remains in each actual
    realization. No per-sample/channel-slot/RX-power normalization is performed.
    """
    channel = asdict(cfg.channel)
    key = _canonical([channel, asdict(cfg.antennas), asdict(cfg.carrier), profile, float(ds)])
    if key not in _CDL_POWER_REFERENCES:
        count = cfg.channel.power_calibration_samples
        seeds = np.random.SeedSequence(cfg.channel.power_calibration_seed).generate_state(count)
        total = 0.0
        band_power = None
        for seed in seeds:
            ch = _raw_cdl_channel(cfg, _bwp(cfg), profile, ds, 0.0, int(seed))
            if band_power is None:
                response = np.fft.fftshift(np.fft.fft(ch.getCoeffMatrix(), n=ch.bwp.nFFT, axis=-1), axes=-1)
                k0 = (ch.bwp.nFFT-12*cfg.carrier.n_rbs)//2
                band_power = np.mean(np.abs(response[:, k0:k0+12*cfg.carrier.n_rbs])**2, axis=-1)
            rays, _ = _cdl_spatial_rays(ch)
            # [ray,Rx,Tx,path] -> average power per physical H coefficient
            total += float((np.abs(rays)**2*band_power).sum(axis=(0, 3)).mean())
        power = total/count
        if not np.isfinite(power) or power <= 0:
            raise ValueError('CDL average-power calibration is nonpositive or nonfinite.')
        _CDL_POWER_REFERENCES[key] = dict(mode='unit_ensemble_average', target_power=1.0,
            reference_power=power, amplitude_scale=float(1/np.sqrt(power)),
            calibration_samples=count, calibration_seed=cfg.channel.power_calibration_seed,
            reference='mean E|H(k,Rx,Tx)|^2 over the active BWP and physical antenna pairs; before W',
            per_sample_normalization=False)
    return dict(_CDL_POWER_REFERENCES[key])

def _make_channel(cfg, bwp, profile, ds, fd, seed):
    ch = _raw_cdl_channel(cfg, bwp, profile, ds, fd, seed)
    info = (_cdl_power_reference(cfg, profile, ds) if cfg.channel.normalize_average_power
            else dict(mode='none', amplitude_scale=1.0, per_sample_normalization=False))
    ch.average_power_scale = info['amplitude_scale']
    ch.average_power_normalization = info
    return ch


"""Notebook-local batched CDL kernel; this file is embedded, never imported at runtime."""

class CdlBatchKernel:
    """Cache fixed geometry once; evaluate random polarization/rays in a GPU batch.

    NumPy draws retain NeoRadium's exact per-sample seed sequence. Antenna fields
    for the 20x20 possible angle pairs are invariant, so compute them once.
    """
    def __init__(self,sim,condition):
        from neoradium.cdl import rayOffsets
        self.sim=sim
        self.channel=_make_channel(sim.cfg,sim.bwp,condition['profile'],
            condition['delay_spread_ns'],condition['doppler_hz'],0)
        ch=self.channel;offset=int(ch.hasLos)
        self.n=len(ch.aods)-offset
        spreads=ch.angleSpreads
        phiD,phiA,thetaD,thetaA=[v[offset:,None]+s*np.asarray(rayOffsets)
            for v,s in zip((ch.aods,ch.aoas,ch.zods,ch.zoas),spreads)]
        if ch.scalingAngleMeans is not None:
            phiD,phiA,thetaD,thetaA=ch.applyAngleScaling(phiD,phiA,thetaD,thetaA,
                10.**(ch.pathPowers[offset:]/10))
        phiD=ch.wrapAngles(phiD,'-180,180');phiA=ch.wrapAngles(phiA,'-180,180')
        thetaD=ch.wrapAngles(thetaD,'0,180');thetaA=ch.wrapAngles(thetaA,'0,180')
        shape=(self.n,20,20)
        tx_theta=np.broadcast_to(thetaD[:,None,:],shape)
        tx_phi=np.broadcast_to(phiD[:,:,None],shape)
        rx_theta=np.broadcast_to(thetaA[:,:,None],shape)
        rx_phi=np.broadcast_to(phiA[:,None,:],shape)
        def fields(antenna,theta,phi,orientation):
            f,l=antenna.getElementsFields(theta.reshape(self.n,400),phi.reshape(self.n,400),orientation)
            return f.reshape(*shape,-1,2),l.reshape(*shape,-1)
        tx,txloc=fields(ch.txAntenna,tx_theta,tx_phi,ch.txOrientation)
        rx,rxloc=fields(ch.rxAntenna,rx_theta,rx_phi,ch.rxOrientation)
        az,zen=np.deg2rad(ch.ueDirAZ)
        velocity=ch.dopplerShift*np.array([np.sin(zen)*np.cos(az),np.sin(zen)*np.sin(az),np.cos(zen)])
        t,p=np.deg2rad(rx_theta),np.deg2rad(rx_phi)
        direction=np.stack([np.sin(t)*np.cos(p),np.sin(t)*np.sin(p),np.cos(t)],axis=-1)
        frequency=(direction*velocity).sum(-1)
        device=sim.device
        self.tx=torch.as_tensor(tx,dtype=torch.complex128,device=device)
        self.rx=torch.as_tensor(rx,dtype=torch.complex128,device=device)
        self.txloc=torch.as_tensor(txloc,dtype=torch.complex128,device=device)
        self.rxloc=torch.as_tensor(rxloc,dtype=torch.complex128,device=device)
        self.frequency=torch.as_tensor(frequency,dtype=torch.float64,device=device)
        self.cluster=torch.arange(self.n,device=device)[None,:,None]
        self.ray=torch.arange(20,device=device)[None,None,:]
        kappa=10.**(ch.xPolPower/10)
        self.polarization=torch.as_tensor(np.sqrt([[1,1/kappa],[1/kappa,1]]),device=device)
        self.path_scale=torch.as_tensor(np.sqrt(10.**(ch.pathPowers[offset:]/10)/20),device=device)
        self.normalization=(np.sqrt(ch.nrNt[0]) if ch.normalizeOutput else 1.)
        self.gain_normalization=np.sqrt((10.**(ch.pathPowers/10)).sum()) if ch.normalizeGains else 1.
        self.coeff=ch.getCoeffMatrix()
        if self.coeff.shape[1]-1>=sim.nfft:raise ValueError('Channel FIR exceeds FFT length.')
        self.los=None
        if ch.hasLos:
            spatial,frequencies=_cdl_spatial_rays(ch)
            self.los=torch.as_tensor(spatial[0,:,:,0],device=device)
            self.los_frequency=float(frequencies[0,0])

    def draw(self,seeds):
        # Keep the existing CPU random stream, including all ray permutations.
        # Channel matrix arithmetic below runs on the selected CUDA device.
        count=len(seeds);coupling=np.empty((count,3,self.n,20),np.int64)
        phases=np.empty((count,self.n,20,2,2),np.float64)
        for b,seed in enumerate(seeds):
            rng=np.random.default_rng(int(seed[0]))
            coupling[b]=[[rng.choice(range(20),size=20,replace=False)
                          for _ in range(self.n)] for _ in range(3)]
            phases[b]=(2*np.pi*rng.random((2,2,self.n,20))-np.pi).transpose(2,3,0,1)
        c=torch.as_tensor(coupling,device=self.sim.device)
        rx=self.rx[self.cluster,c[:,1],c[:,0]]
        tx=self.tx[self.cluster,self.ray,c[:,2]]
        rxloc=self.rxloc[self.cluster,c[:,1],c[:,0]]
        txloc=self.txloc[self.cluster,self.ray,c[:,2]]
        phase=torch.as_tensor(phases,device=self.sim.device)
        pol=torch.exp(1j*phase)*self.polarization
        spatial=(rx@pol@tx.transpose(-2,-1))*rxloc[...,None]*txloc[...,None,:]
        spatial=spatial*self.path_scale[None,:,None,None,None]
        spatial=spatial/self.normalization/self.gain_normalization
        spatial=spatial.permute(0,2,3,4,1)
        frequency=self.frequency[self.cluster,c[:,1],c[:,0]]
        if self.los is not None:
            los=torch.zeros((count,20,self.sim.nr,self.sim.nt,1),dtype=torch.complex128,device=self.sim.device)
            los[:,0,:,:,0]=self.los
            spatial=torch.cat((los,spatial),dim=-1)
            lf=torch.full((count,1,20),self.los_frequency,dtype=torch.float64,device=self.sim.device)
            frequency=torch.cat((lf,frequency),dim=1)
        spatial=(spatial*self.channel.average_power_scale).to(self.sim.dtype)
        frequency=frequency.to(self.sim.rdtype)
        phase=2*np.pi*self.sim.tensor(self.sim.anchors/self.sim.fs,real=True)[None,:,None,None]*frequency[:,None]
        gains=torch.einsum('bmrtp,bapm->bartp',spatial,torch.exp(1j*phase))
        return [self.channel],np.asarray(seeds),gains,self.coeff


def _raw_tdl_channel(cfg,bwp,profile,ds,fd,seed):
    nt,nr=antenna_counts(cfg);c=cfg.channel
    kwargs=dict(delaySpread=ds,dopplerShift=fd,txAntennaCount=nt,rxAntennaCount=nr,
        normalizeGains=c.normalize_gains,normalizeOutput=c.normalize_output,
        filterLen=c.filter_len,delayQuantSize=c.delay_quant_size,seed=int(seed),
        mimoCorrelation=c.tdl_mimo_correlation,sosType='Xiao',sosNumSins=c.tdl_num_sinusoids)
    if c.k_factor_db is not None:
        if profile not in ('D','E'):raise ValueError('K-factor scaling applies to LOS TDL-D/E only.')
        kwargs['kFactor']=c.k_factor_db
    return TdlChannel(bwp,profile=profile,**kwargs)


def _tdl_power_reference(cfg,ch):
    response=np.fft.fftshift(np.fft.fft(ch.getCoeffMatrix(),n=ch.bwp.nFFT,axis=-1),axes=-1)
    k0=(ch.bwp.nFFT-12*cfg.carrier.n_rbs)//2
    band=np.mean(np.abs(response[:,k0:k0+12*cfg.carrier.n_rbs])**2,axis=-1)
    powers=10.**(ch.pathPowers/10)
    if ch.normalizeGains:powers=powers/powers.sum()
    power=float(np.dot(band,powers))/(ch.nrNt[0] if ch.normalizeOutput else 1.)
    if not np.isscalar(ch.correlationMatrix):
        cov=ch.correlationMatrix
        if ch.normalizeGains:cov=cov*cov.shape[0]/np.trace(cov)
        power*=float(np.trace(cov).real/cov.shape[0])
    if not np.isfinite(power) or power<=0:raise ValueError('Invalid TDL ensemble channel power.')
    return dict(mode='unit_ensemble_average',reference_power=power,amplitude_scale=float(1/np.sqrt(power)),
        per_sample_normalization=False,reference='NeoRadium path powers/correlation and active-band fractional-delay FIR; before identity port mapping')


def channel_power_reference(cfg,condition):
    if not cfg.channel.normalize_average_power:return dict(mode='none',amplitude_scale=1.,per_sample_normalization=False)
    if condition['backend'] in ('cost2100','quadriga'):
        nt,nr=antenna_counts(cfg)
        return native_module().power_reference(cfg,condition,nr,nt)
    if condition['backend']=='cdl':return _cdl_power_reference(cfg,condition['profile'],condition['delay_spread_ns'])
    ch=_raw_tdl_channel(cfg,_bwp(cfg),condition['profile'],condition['delay_spread_ns'],0.,0)
    return _tdl_power_reference(cfg,ch)


class TdlBatchKernel:
    """NeoRadium TDL A~E / Xiao SOS, evaluated at all continuous-slot times on CPU/CUDA.

    Profile, delay FIR, MIMO correlation and LOS power come from TdlChannel.
    The initial NumPy draws match NeoRadium's first Xiao realization for each seed.
    One fixed phase draw spans the WHOLE saved sequence; slot boundaries never reset it.
    """
    def __init__(self,sim,condition):
        self.sim=sim
        ch=_raw_tdl_channel(sim.cfg,sim.bwp,condition['profile'],condition['delay_spread_ns'],condition['doppler_hz'],0)
        self.channel=ch;self.coeff=ch.getCoeffMatrix();self.s=ch.sosNumSins;self.p=ch.numPaths
        if self.coeff.shape[1]-1>=sim.nfft:raise ValueError('Channel FIR exceeds FFT length.')
        self.info=channel_power_reference(sim.cfg,condition)
        ch.average_power_normalization=self.info;ch.average_power_scale=self.info['amplitude_scale']
        self.correlation=None
        if not np.isscalar(ch.correlationMatrix):
            cov=np.asarray(ch.correlationMatrix,dtype=np.complex128)
            if ch.normalizeGains:cov=cov*(sim.nr*sim.nt)/np.trace(cov)
            eig,vec=np.linalg.eigh(cov)
            if eig.min() < -1e-8:raise ValueError('TDL spatial correlation matrix is not PSD.')
            self.correlation=torch.as_tensor((vec*np.sqrt(eig.clip(0)))@vec.conj().T,dtype=torch.complex128,device=sim.device)
        powers=np.sqrt(10.**(ch.pathPowers/10))
        if ch.normalizeGains:powers/=np.sqrt((10.**(ch.pathPowers/10)).sum())
        if ch.normalizeOutput:powers/=np.sqrt(sim.nr)
        self.scale=torch.as_tensor(powers*ch.average_power_scale,device=sim.device)

    def draw(self,seeds):
        sim=self.sim;ch=self.channel;n=len(seeds)
        theta=[];phi=[]
        for seed in seeds:
            rng=np.random.default_rng(int(seed[0]))
            # TdlChannel.__init__ allocates these two GMEDS1 phase arrays even in Xiao mode.
            for _ in range(2):rng.random((1,self.s,sim.nr,sim.nt,self.p))
            theta.append(rng.random((self.s,1,1,self.p))*2*np.pi-np.pi)
            phi.append(rng.random((self.s,sim.nr,sim.nt,self.p))*2*np.pi-np.pi)
        theta=torch.as_tensor(np.asarray(theta),device=sim.device)
        phi=torch.as_tensor(np.asarray(phi),device=sim.device)
        order=torch.arange(1,self.s+1,dtype=torch.float64,device=sim.device)[None,:,None,None,None]
        alpha=(2*np.pi*order+theta)/self.s
        times=torch.as_tensor(sim.anchors/sim.fs,dtype=torch.float64,device=sim.device)
        # Keep the temporary bounded by the same generation batch/path limits as CDL.
        omega=2*np.pi*ch.dopplerShift*torch.cos(alpha)
        gains=[]
        for time in times:
            phase=time*omega+phi
            value=torch.exp(1j*phase).sum(1)/np.sqrt(self.s)
            if self.correlation is not None:
                value=torch.einsum('bip,ij->bjp',value.reshape(n,sim.nr*sim.nt,self.p),self.correlation).reshape(n,sim.nr,sim.nt,self.p)
            if ch.hasLos:
                k=10.**(ch.kFactorLos/10)
                value[:,:,:,0]=(value[:,:,:,0]+np.sqrt(k)*torch.exp(2j*np.pi*ch.losDopplerShift*time))/np.sqrt(k+1)
            gains.append(value*self.scale)
        return [ch],np.asarray(seeds),torch.stack(gains,1).to(sim.dtype),self.coeff


## 포트별 RS 배치 · SISO 독립 LS / 다중 포트 CDM LS · Oracle LMMSE


In [ ]:
def _positive_integer(value, name):
    if isinstance(value,(bool,np.bool_)) or not np.isfinite(value) or int(value)!=value or value<=0:
        raise ValueError(f'{name} must be a positive integer.')
    return int(value)


def antenna_counts(cfg):
    a=cfg.antennas
    return (_antenna_ports(a.tx_shape,a.tx_polarization),
            _antenna_ports(a.rx_shape,a.rx_polarization))


def antenna_mode(nt,nr):
    if nt==1 and nr==1:return 'SISO'
    kind='MISO' if nr==1 else 'SIMO' if nt==1 else 'MIMO'
    return f'{kind} ({nt}x{nr})'


def validate_config(cfg):
    if neoradium.__version__!='0.5.3':
        raise RuntimeError('This notebook is verified with neoradium==0.5.3.')
    if cfg.carrier.n_rbs!=10 or tuple(cfg.dmrs_symbols)!=(2,11):
        raise ValueError('Use 10 RB / 60 RS per port at symbols (2,11).')
    if cfg.carrier.cp_type.lower()!='normal' or cfg.carrier.scs_khz!=15:
        raise ValueError('Use normal CP and SCS=15 kHz.')
    nt,nr=antenna_counts(cfg)
    # Type-1 single-symbol DM-RS has four orthogonal ports at each RS symbol.
    if nt not in (1,2,3,4) or nr<1:
        raise ValueError('This two-symbol Type-1 allocation supports Tx 1~4, Rx >=1.')
    if len(cfg.split_counts)!=3 or any(isinstance(n,(bool,np.bool_)) or not isinstance(n,(int,np.integer)) or n<0 for n in cfg.split_counts):
        raise ValueError('Provide nonnegative integer Train/Val/Test counts.')
    if cfg.sample_count<1:raise ValueError('At least one split must contain samples.')
    if not isinstance(cfg.interpolate_2d,bool) or cfg.compute_dtype not in ('complex64','complex128'):
        raise ValueError('Use a boolean interpolation switch and a complex compute dtype.')
    if not np.isfinite(cfg.tx_epre) or cfg.tx_epre<=0:raise ValueError('tx_epre must be positive.')
    _positive_integer(cfg.n_slots,'n_slots')
    _positive_integer(cfg.channel.tdl_num_sinusoids,'tdl_num_sinusoids')
    Modem(cfg.modulation)
    return _bwp(cfg)


def linear_weights(known, query):
    known=np.asarray(known,dtype=float);query=np.asarray(query,dtype=float)
    if known.ndim!=1 or len(known)<2 or np.any(np.diff(known)<=0):
        raise ValueError('Interpolation nodes must be strictly increasing.')
    if query.ndim!=1 or np.any(query<known[0]) or np.any(query>known[-1]):
        raise ValueError('Interpolation queries must stay inside the first / last RS.')
    hi=np.searchsorted(known,query,side='right').clip(1,len(known)-1);lo=hi-1
    alpha=(query-known[lo])/(known[hi]-known[lo])
    weights=np.zeros((len(query),len(known)),dtype=np.float64)
    weights[np.arange(len(query)),lo]=1-alpha;weights[np.arange(len(query)),hi]+=alpha
    return weights


def build_dmrs_layout(cfg,slot=0):
    bwp=validate_config(cfg);nt,nr=antenna_counts(cfg)
    if isinstance(slot,(bool,np.bool_)) or not isinstance(slot,(int,np.integer)) or not 0<=slot<cfg.n_slots:
        raise ValueError('slot is outside the saved range.')
    bwp.carrier.slotNo=int(slot);bwp.carrier.frameNo=int(slot)//bwp.slotsPerFrame
    pdsch=NrPDSCH(bwp,mappingType='A',numLayers=nt,modulation=cfg.modulation,
        prbSet=list(range(cfg.carrier.n_rbs)),symStart=0,symLen=14,rnti=1,nID=1,portSet=list(range(nt)))
    pdsch.setDMRS(configType=1,symbols=1,typeA1stPos=2,additionalPos=1,nIDs=[1,1],scID=0)
    pdsch.initGrid()
    # NeoRadium uses the OFDM symbol index in the DM-RS sequence initialization.
    # Thus symbols 2 and 11 use different QPSK sequences at the SAME SC positions per port.
    pilot_grid=np.zeros((nt,14,120),np.complex128)
    indices=pdsch.grid.getReIndexes('DMRS')
    pilot_grid[indices]=pdsch.grid.grid[indices]*np.sqrt(cfg.tx_epre)
    pilot_masks=np.abs(pilot_grid)>0;pilot_mask=pilot_masks.any(0)
    symbols=np.flatnonzero(pilot_mask.any(1));rs_k=[]
    if not np.array_equal(symbols,cfg.dmrs_symbols):raise ValueError('Unexpected DM-RS symbols.')
    for port in range(nt):
        a,b=[np.flatnonzero(pilot_masks[port,l]) for l in symbols]
        if len(a)!=60 or not np.array_equal(a,b):raise ValueError('Expected aligned 60-RS positions for each port.')
        rs_k.append(a)
    rs_k=np.stack(rs_k)
    # The Data/interpolation region is the intersection of all port frequency intervals.
    # SISO and ports [0,1]: k=0..118, 119 subcarriers. Ports [0..2/3]: k=1..118, 118 subcarriers.
    # Raw RS are saved at all 60 coordinates of EACH port. Real covers the union,
    # so every port's edge RS also has a physical ground-truth value.
    full_k=np.arange(rs_k[:,0].max(),rs_k[:,-1].min()+1)
    full_l=np.arange(symbols[0],symbols[-1]+1)
    out_k=full_k if cfg.interpolate_2d else rs_k[0].copy()
    out_l=full_l if cfg.interpolate_2d else symbols.copy()
    data_mask=np.zeros_like(pilot_mask)
    data_mask[np.ix_(np.arange(symbols[0]+1,symbols[-1]),full_k)]=True
    observations=[];inverses=[];groups=[];assignment=np.full((nt,60,2),-1,np.int64)
    if nt==1:
        # One observation per RE: NO FD-CDM grouping/averaging.
        for li,l in enumerate(symbols):
            for ki,k in enumerate(rs_k[0]):
                q=len(observations);observations.append([[l,k]])
                inverses.append(np.asarray([[1/pilot_grid[0,l,k]]]))
                groups.append([0]);assignment[0,ki,li]=q
    else:
        for row in pdsch.dmrs.getCdmLKs():
            for bundle in row:
                ell,kay=bundle.T
                ports=np.flatnonzero(np.any(pilot_masks[:,ell,kay],axis=1))
                matrix=pilot_grid[ports[:,None],ell[None],kay[None]].T
                if np.linalg.matrix_rank(matrix)!=len(ports):raise ValueError('DM-RS CDM bundle is not full rank.')
                q=len(observations);observations.append(bundle);inverses.append(np.linalg.pinv(matrix));groups.append(ports.tolist())
                li=int(np.searchsorted(symbols,ell[0]))
                for p in ports:assignment[p,np.searchsorted(rs_k[p],kay),li]=q
    if np.any(assignment<0):raise ValueError('Unassigned DM-RS positions.')
    nobs=len(observations);width=len(observations[0]);inverse=np.zeros((nobs,nt,width),np.complex128)
    matrices=np.zeros((nobs,width,nt),np.complex128)
    for q,(bundle,inv,ports) in enumerate(zip(observations,inverses,groups)):
        ports=np.asarray(ports,dtype=np.int64)
        inverse[q,ports]=inv
        ell,kay=np.asarray(bundle).T
        matrices[q,:,ports]=pilot_grid[ports[:,None],ell[None],kay[None]]
    observations=np.asarray(observations)
    return dict(bwp=bwp,pilot_grid=pilot_grid,pilot_masks=pilot_masks,pilot_mask=pilot_mask,
        data_mask=data_mask,rs_k=rs_k[0],rs_k_by_port=rs_k,rs_l=symbols,
        output_k=out_k,output_l=out_l,truth_k=np.arange(rs_k[:,0].min(),rs_k[:,-1].max()+1),truth_l=full_l,data_k=full_k,
        output_rs_mask=pilot_mask[np.ix_(out_l,out_k)].T,
        output_data_mask=data_mask[np.ix_(out_l,out_k)].T,
        frequency_weights=np.stack([linear_weights(k,full_k if cfg.interpolate_2d else k) for k in rs_k]),
        time_weights=linear_weights(symbols,out_l),observation_l=observations[:,:,0],
        observation_k=observations[:,:,1],inverse=inverse,pilot_matrices=matrices,
        observation_li=np.searchsorted(symbols,observations[:,0,0]),assignment=assignment,
        cdm_enabled=nt>1,nt=nt,nr=nr)


def interpolate_channel(pilot_channel,layout):
    # [batch,port,rx,60,2] -> [batch,port,rx,K,L]. No mixing of samples/ports/Rx.
    if pilot_channel.shape[-2:]!=(60,2):raise ValueError('Expected 60 x 2 RS values.')
    f=torch.as_tensor(layout['frequency_weights'],dtype=pilot_channel.dtype,device=pilot_channel.device)
    t=torch.as_tensor(layout['time_weights'],dtype=pilot_channel.dtype,device=pilot_channel.device)
    return torch.einsum('pqk,bprkl,tl->bprqt',f,pilot_channel,t)


def estimate_pilot_channels(rx,real_grid,layout,noise_variance_re):
    # rx [B,Rx,L,K], real_grid [B,port,Rx,L,K]. Real is used ONLY by oracle LMMSE.
    device=rx.device;dtype=rx.dtype
    ell=torch.as_tensor(layout['observation_l'],device=device)
    kay=torch.as_tensor(layout['observation_k'],device=device)
    inv=torch.as_tensor(layout['inverse'],dtype=dtype,device=device)
    received=rx[:,:,ell,kay]                           # B,Rx,Q,bundle
    obs=torch.einsum('brqw,qpw->brqp',received,inv)
    ls=rx.new_empty((len(rx),layout['nt'],rx.shape[1],60,2))
    assignment=layout['assignment']
    for p in range(layout['nt']):ls[:,p]=obs[:,:,torch.as_tensor(assignment[p],device=device),p]
    # Same rank-one true-channel covariance LMMSE definition as the supplied SRS notebook.
    # Each symbol/Rx uses all its received RS REs, including the port-specific pilot sequence.
    # Multi-port covariance is block-diagonal across ports; each block is h_p h_p^H.
    # This is explicitly an oracle baseline, not an estimated statistical prior.
    mmse=torch.empty_like(ls)
    for li,l in enumerate(layout['rs_l']):
        ks=np.unique(layout['observation_k'][layout['observation_li']==li])
        k=torch.as_tensor(ks,device=device)
        y=rx[:,:,int(l),k]                             # B,Rx,M
        h=real_grid[:,:,:,int(l),k].permute(0,2,3,1)   # B,Rx,M,P
        pilots=torch.as_tensor(layout['pilot_grid'][:,int(l),ks].T,dtype=dtype,device=device)
        u=h*pilots[None,None]
        gram=u.mH@u
        n0=float(noise_variance_re)
        eye=torch.eye(layout['nt'],dtype=dtype,device=device)
        rhs=u.mH@y[...,None]
        # P x P solve only (P<=4), rather than an M x M inverse at every sample.
        if n0>0:coeff=torch.linalg.solve(gram+n0*eye,rhs)[...,0]
        else:coeff=(torch.linalg.pinv(u)@y[...,None])[...,0]
        for p,positions in enumerate(layout['rs_k_by_port']):
            kp=torch.as_tensor(positions,device=device)
            mmse[:,p,:,:,li]=real_grid[:,p,:,int(l),kp]*coeff[:,:,p,None]
    return ls,mmse


CONFIG,EXECUTION=make_notebook_settings()


## CP-OFDM · TDL/CDL · AWGN — CPU/CUDA 배치 실행


In [ ]:
class DmrsChannelBatch:
    def __init__(self,cfg,settings):
        self.cfg=copy.deepcopy(cfg);self.settings=copy.deepcopy(settings)
        self.layouts=[build_dmrs_layout(cfg,slot) for slot in range(_positive_integer(cfg.n_slots,'n_slots'))]
        self.layout=self.layouts[0];self.bwp=self.layout['bwp']
        self.n_slots=int(cfg.n_slots)
        self.pilot_grid=np.concatenate([layout['pilot_grid'] for layout in self.layouts],axis=1)
        self.data_mask=np.concatenate([layout['data_mask'] for layout in self.layouts])
        self.nt,self.nr=antenna_counts(cfg)
        scale=self.n_slots*self.nt*self.nr
        self.effective_batch_size=max(1,settings.batch_size//(1<<(scale-1).bit_length()))
        device=settings.device
        self.device=torch.device(('cuda' if torch.cuda.is_available() else 'cpu') if device=='auto' else device)
        if self.device.type=='cuda' and not torch.cuda.is_available():
            raise RuntimeError('CUDA is unavailable; select COMPUTE_DEVICE="cpu" or "auto".')
        _positive_integer(settings.batch_size,'batch_size')
        _positive_integer(settings.path_chunk_size,'path_chunk_size')
        self.dtype=getattr(torch,cfg.compute_dtype)
        self.rdtype=torch.float32 if self.dtype==torch.complex64 else torch.float64
        self.nsym=14*self.n_slots;self.nk=120
        self.nfft=self.bwp.nFFT;self.fs=self.bwp.sampleRate
        self.modem=Modem(cfg.modulation)
        slot_lengths=[layout['bwp'].getSymLens().copy() for layout in self.layouts]
        self.slot_samples=np.asarray([int(value[:-1].sum()) for value in slot_lengths])
        self.slot_starts=np.r_[0,np.cumsum(self.slot_samples[:-1])]
        lengths=np.concatenate([value[:-1] for value in slot_lengths]+[slot_lengths[-1][-1:]])
        cp=lengths[:-1]-self.nfft
        starts=np.r_[0,np.cumsum(lengths[:-2])]+cp
        count=int(lengths[:-1].sum())
        self.slot_duration_s=float(self.slot_samples[0])/self.fs
        self.sequence_duration_s=count/self.fs
        self.symbol_starts=starts.copy()
        self.anchors=np.r_[starts,count+lengths[-1]-self.nfft].astype(float)
        cp_indices=np.concatenate([(np.arange(length)-c)%self.nfft+i*self.nfft
            for i,(length,c) in enumerate(zip(lengths[:-1],cp))])
        self.plan=dict(n=count,cp_indices=self.tensor(cp_indices,integer=True),
            fft_indices=self.tensor(starts[:,None]+np.arange(self.nfft),integer=True),intervals=[])
        times=np.arange(count)
        idx=np.searchsorted(self.anchors,times,side='right').clip(1,len(self.anchors)-1)-1
        cuts=np.r_[0,np.flatnonzero(np.diff(idx))+1,count]
        for lo,hi in zip(cuts[:-1],cuts[1:]):
            j=int(idx[lo]);a=((times[lo:hi]-self.anchors[j])/(self.anchors[j+1]-self.anchors[j])).clip(0,1)
            self.plan['intervals'].append((int(lo),int(hi),j,self.tensor(a,real=True)))
        weights=np.zeros((self.nsym,len(self.anchors)))
        for l,start in enumerate(starts):
            t=start+np.arange(self.nfft)
            j=np.searchsorted(self.anchors,t,side='right').clip(1,len(self.anchors)-1)-1
            a=((t-self.anchors[j])/(self.anchors[j+1]-self.anchors[j])).clip(0,1)
            weights[l]=np.bincount(j,weights=1-a,minlength=len(self.anchors))
            weights[l]+=np.bincount(j+1,weights=a,minlength=len(self.anchors))
        self.mean_weights=self.tensor(weights/self.nfft)
        self.filters={}
        self.cdl_kernels={}
        self.sample_maps={}

    def source_sample_indices(self,condition):
        if self.cfg.source_split_counts is None:return np.arange(self.cfg.sample_count,dtype=np.int64)
        identity=[self.cfg.source_selection_seed,condition]
        key=_canonical(identity)
        if key not in self.sample_maps:
            words=np.frombuffer(hashlib.sha256(key.encode()).digest(),dtype='<u4')
            rng=np.random.default_rng(np.random.SeedSequence(words));parts=[];start=0
            for available,take in zip(self.cfg.source_split_counts,self.cfg.split_counts):
                if take>available:raise ValueError('Mix source quota exceeds its original partition.')
                selected=np.arange(available) if take==available else np.sort(rng.choice(available,take,replace=False))
                parts.append(selected+start);start+=available
            self.sample_maps[key]=np.concatenate(parts).astype(np.int64)
        return self.sample_maps[key]

    def sample_seeds(self,condition,index):
        # SNR / interpolation / device / batch size do not change the physical draw.
        if self.cfg.source_split_counts is not None:index=int(self.source_sample_indices(condition)[int(index)])
        identity=([int(self.cfg.seed),condition['backend'],condition['profile'],condition['native_settings'],int(index)]
            if condition['backend'] in ('cost2100','quadriga') else
            [int(self.cfg.seed),condition['backend'],condition['profile'],float(condition['delay_spread_ns']),float(condition['doppler_hz']),int(index)])
        words=np.frombuffer(hashlib.sha256(_canonical(identity).encode()).digest(),dtype='<u4')
        return np.random.SeedSequence(words).generate_state(3)

    def _channel_batch(self,condition,indices):
        key=(_canonical({k:v for k,v in condition.items() if k!='snr_db'}) if condition['backend'] in ('cost2100','quadriga')
             else tuple(condition[k] for k in ('backend','profile','delay_spread_ns','doppler_hz')))
        if key not in self.cdl_kernels:
            kernel=(native_module().NativeBatchKernel if condition['backend'] in ('cost2100','quadriga')
                    else CdlBatchKernel if condition['backend']=='cdl' else TdlBatchKernel)
            self.cdl_kernels[key]=kernel(self,condition)
        seeds=[self.sample_seeds(condition,index) for index in indices]
        return self.cdl_kernels[key].draw(seeds)

    @torch.inference_mode()
    def simulate(self,condition,indices,*,debug=False,coded_data_bits=None):
        validate_condition(condition)
        indices=[int(i) for i in indices]
        if not indices or any(i<0 for i in indices):raise ValueError('Provide nonnegative sample indices.')
        batch=len(indices);cfg=self.cfg
        channels,seeds,gains,coeff=self._channel_batch(condition,indices)
        native_cir=coeff is None
        memory=gains.shape[-1]-1 if native_cir else coeff.shape[1]-1
        grid=np.broadcast_to(self.pilot_grid,(batch,self.nt,self.nsym,120)).copy()
        data_mask=np.broadcast_to(self.data_mask,(self.nt,self.nsym,120))
        data_bits=[]
        bits_per_sample=int(data_mask.sum())*self.modem.qm
        if coded_data_bits is not None:
            coded_data_bits=np.asarray(coded_data_bits)
            if coded_data_bits.shape!=(batch,bits_per_sample) or not np.isin(coded_data_bits,[0,1]).all():
                raise ValueError('coded_data_bits must contain all saved slots for each sample.')
        for b,seed in enumerate(seeds):
            bits=np.random.default_rng(int(seed[1])).integers(0,2,bits_per_sample,dtype=np.int8)
            if coded_data_bits is not None:bits=coded_data_bits[b].astype(np.int8,copy=False)
            grid[b][data_mask]=self.modem.modulate(bits)*np.sqrt(cfg.tx_epre)
            if debug:data_bits.append(bits)
        # One continuous waveform/convolution: earlier-slot Tx samples remain in FIR history.
        wave=self._ofdm(self.tensor(grid),self.plan)
        extended=torch.nn.functional.pad(wave,(memory,0))
        clean=(native_module().apply_cir_waveform(wave,gains,self.plan) if native_cir
               else self._filtered_mix(extended,gains,coeff,self.plan,self.plan['n']))
        noise_variance=cfg.tx_epre*10.**(-condition['snr_db']/10) if cfg.noise_enabled else 0.
        if noise_variance:
            noise=[]
            for seed in seeds:
                rng=np.random.default_rng(int(seed[2]))
                # Slot 0 uses the former one-slot real/imag random sequence exactly.
                noise.append(np.concatenate([rng.standard_normal((2,self.nr,int(n))) for n in self.slot_samples],axis=-1))
            noise=np.stack(noise)
            clean+=self.tensor((noise[:,0]+1j*noise[:,1]))*np.sqrt(noise_variance/self.nfft/2)
        k0=(self.nfft-self.nk)//2
        phase=self.tensor(np.exp(2j*np.pi*(np.arange(self.nk)-self.nk//2)*int(channels[0].filterDelay)/self.nfft)
            if cfg.channel.compensate_fir_latency else np.ones(self.nk))
        rx=torch.fft.fftshift(torch.fft.fft(clean[:,:,self.plan['fft_indices']],dim=-1),dim=-1)
        rx=rx[:,:,:,k0:k0+self.nk]*phase
        if native_cir:
            real_grid=native_module().cir_real_channel(gains,self.mean_weights,self.nfft,self.nk,
                int(channels[0].filterDelay) if cfg.channel.compensate_fir_latency else 0)
        else:
            response=torch.fft.fftshift(torch.fft.fft(self.tensor(coeff),n=self.nfft,dim=-1),dim=-1)
            k0=(self.nfft-self.nk)//2;response=response[:,k0:k0+self.nk]
            response=response*phase
            mean=torch.einsum('sa,bartp->bsrtp',self.mean_weights,gains)
            real_grid=torch.einsum('bsrtp,pk->btrsk',mean,response)
        values={'ls':[],'mmse':[],'real':[]};pilot_ls_values=[];pilot_mmse_values=[]
        for slot,layout in enumerate(self.layouts):
            lo,hi=14*slot,14*(slot+1)
            pilot_ls,pilot_mmse=estimate_pilot_channels(rx[:,:,lo:hi],real_grid[:,:,:,lo:hi],layout,noise_variance)
            values['ls'].append(interpolate_channel(pilot_ls,layout))
            values['mmse'].append(interpolate_channel(pilot_mmse,layout))
            # Real is the physical CFR at every saved RE, independently for every slot.
            ell=torch.as_tensor(layout['truth_l'],device=self.device)
            kay=torch.as_tensor(layout['truth_k'],device=self.device)
            values['real'].append(real_grid[:,:,:,lo:hi][:,:,:,ell[:,None],kay[None,:]].transpose(-1,-2))
            if debug:pilot_ls_values.append(pilot_ls);pilot_mmse_values.append(pilot_mmse)
        result={}
        for key,slots in values.items():
            value=torch.stack(slots,dim=1)
            if not torch.isfinite(value).all():raise FloatingPointError(f'Nonfinite {key} channel.')
            result[key]=value.to(getattr(torch,cfg.compute_dtype)).cpu().numpy()
        result.update(indices=np.asarray(indices,dtype=np.int64),seeds=seeds,
            noise_variance_re=float(noise_variance),power_normalization=channels[0].average_power_normalization)
        if debug:
            result['debug']=dict(tx_grid=grid,rx_grid=rx.cpu().numpy(),real_grid=real_grid.cpu().numpy(),
                data_bits=np.stack(data_bits),pilot_ls=torch.stack(pilot_ls_values,dim=1).cpu().numpy(),
                pilot_mmse=torch.stack(pilot_mmse_values,dim=1).cpu().numpy(),
                slot_start_times_s=self.slot_starts/self.fs,symbol_fft_start_times_s=self.symbol_starts/self.fs)
        return result

    def clear_device_cache(self):
            if self.device.type=='cuda':torch.cuda.empty_cache()

    def tensor(self,value,integer=False,real=False):
            return torch.as_tensor(value,dtype=torch.long if integer else self.rdtype if real else self.dtype,
                device=self.device)

    def _ofdm(self,grid,plan):
            padded=torch.nn.functional.pad(grid,((self.nfft-self.nk)//2,)*2)
            symbols=torch.fft.ifft(torch.fft.ifftshift(padded,dim=-1),dim=-1)
            return symbols.flatten(2)[:,:,plan['cp_indices']]

    def _filtered_mix(self,extended,gains,coeff,plan,n):
            """Exact linear convolution, then linear complex gain interpolation.

            FFT padding prevents circular wrap. Physical Tx history is supplied even
            when current data use another W. Path chunks bound VRAM use.
            """
            memory=coeff.shape[1]-1
            size=1 << (extended.shape[-1]+memory-1).bit_length()
            key=(size,coeff.shape,coeff.tobytes())
            if key not in self.filters:self.filters[key]=torch.fft.fft(self.tensor(coeff),n=size,dim=-1)
            spectrum=torch.fft.fft(extended,n=size,dim=-1)
            output=torch.zeros((len(extended),self.nr,n),dtype=self.dtype,device=self.device)
            for p in range(0,len(coeff),self.settings.path_chunk_size):
                count=min(self.settings.path_chunk_size,len(coeff)-p)
                filtered=torch.fft.ifft(spectrum[:,None]*self.filters[key][None,p:p+count,None],dim=-1)
                filtered=filtered[:,:,:,memory:memory+n]
                for lo,hi,j,a in plan['intervals']:
                    hi=min(hi,n)
                    if hi<=lo:continue
                    block=filtered[:,:,:,lo:hi]
                    left=torch.einsum('brtc,bctn->brn',gains[:,j,:,:,p:p+count],block)
                    right=torch.einsum('brtc,bctn->brn',gains[:,j+1,:,:,p:p+count],block)
                    alpha=a[:hi-lo]
                    output[:,:,lo:hi]+=(1-alpha)*left+alpha*right
            return output


## 저장 구조

DATA_ROOT / SISO·MISO·SIMO·MIMO / Fix 또는 Mix / 기존 채널·RS 배치 양식.
Fix와 Mix 파일은 수량 및 split 구조가 다르므로 모드 폴더를 분리합니다.
CDM은 경로에 넣지 않고 파일 속성에 기록합니다.


In [ ]:
def native_module():
    import sys
    directory=Path(r'C:/Users/ychan/Desktop/VScode/DMRS_CSI_Compression/GPT Output/native_channel_implementation_20261103')
    if str(directory) not in sys.path:sys.path.insert(0,str(directory))
    import native_kernel
    digest=native_kernel.implementation_digest()
    if getattr(native_kernel,'_loaded_digest',digest)!=digest:
        raise RuntimeError('Native Python source changed after import. Restart this notebook kernel before generating data.')
    native_kernel._loaded_digest=digest
    return native_kernel

def native_condition_settings(name,scenario):
    if name=='COST2100':
        value=copy.deepcopy(COST2100_SETTINGS[scenario]);value['network']=scenario
        if value.get('link','Single')!='Single':raise ValueError('Python COST port currently requires one BS/one MS (Single link).')
        if N_TX>2 or N_RX>2:raise ValueError('Official COST omni mapping currently supports up to 2 Tx / 2 Rx in this port.')
        return value
    value=copy.deepcopy(QUADRIGA_SETTINGS)
    value.update(copy.deepcopy(QUADRIGA_SCENARIO_SETTINGS.get(scenario,{})))
    value['scenario']=scenario
    value.setdefault('absolute_delays',True)
    from quadriga_python import validate_settings
    validate_settings(value)
    return value

def condition_text(c):
    if c['backend'] in ('tdl','cdl'):
        return f"{c['backend'].upper()}-{c['profile']} | Delay {c['delay_spread_ns']:g} ns | Doppler {c['doppler_hz']:g} Hz | SNR {c['snr_db']:g} dB"
    return f"{c['backend'].upper()} / {c['profile']} | native geometry/velocity | SNR {c['snr_db']:g} dB"

def condition_folder(c):
    if c['backend'] in ('tdl','cdl'):return c['backend'].upper()+'-'+c['profile']
    family='COST2100' if c['backend']=='cost2100' else 'QuaDRiGa'
    return str(Path(family)/c['profile'])

def condition_sort_key(c):
    return (c['backend'],c['profile'],float(c.get('delay_spread_ns',0)),float(c.get('doppler_hz',0)),float(c['snr_db']))

REFERENCE_IMPLEMENTATIONS = ('8d17b2c5631f6253e3e996a90fa4e1dcf8694751f0a9585f31070e315f501d06',)  # 기존 묶음 LS 데이터는 새 RS별 LS 데이터로 재사용하지 않습니다.

def _compatible_generation_config(saved,cfg):
    saved=copy.deepcopy(saved)
    saved.setdefault('source_split_counts',None)
    saved.setdefault('source_selection_seed',42)
    saved.setdefault('n_slots',1)
    return saved==_jsonable(asdict(cfg))

def validate_condition(c):
    if not np.isfinite(c.get('snr_db',np.nan)):raise ValueError('Invalid SNR.')
    if c.get('backend') in ('cost2100','quadriga'):
        if not isinstance(c.get('native_settings'),dict) or not c.get('profile'):raise ValueError('Native scenario settings are required.')
        return
    if c.get('backend') not in ('tdl','cdl') or c.get('profile') not in 'ABCDE':raise ValueError('Invalid TDL/CDL condition.')
    if not np.isfinite(c.get('delay_spread_ns',np.nan)) or not np.isfinite(c.get('doppler_hz',np.nan)):
        raise ValueError('Finite Delay/Doppler required.')
    if c['delay_spread_ns']<=0 or c['doppler_hz']<0:raise ValueError('Invalid Delay/Doppler.')


def make_conditions(profiles,delays,dopplers,snrs):
    profiles=validate_channel_selection(profiles)
    if not profiles or not snrs:raise ValueError('Channel/SNR lists cannot be empty.')
    output=[]
    for name in profiles:
        if name.startswith(('TDL-','CDL-')):
            if not delays or not dopplers:raise ValueError('TDL/CDL needs Delay/Doppler lists.')
            backend,profile=name.split('-')
            values=[dict(backend=backend.lower(),profile=profile,delay_spread_ns=float(d),doppler_hz=float(v),snr_db=float(s))
                for d,v,s in product(delays,dopplers,snrs)]
        else:
            native_module()
            scenarios=COST2100_SCENARIO_LIST if name=='COST2100' else QUADRIGA_SCENARIO_LIST
            values=[dict(backend='cost2100' if name=='COST2100' else 'quadriga',profile=scene,
                native_settings=native_condition_settings(name,scene),snr_db=float(s)) for scene,s in product(scenarios,snrs)]
        for c in values:
            validate_condition(c)
            if c in output:raise ValueError('Duplicate native/environment condition.')
            output.append(c)
    return output


def _implementation_id():
    # Fingerprint executable semantics, independent of notebook cell counters / filenames.
    import types,inspect
    def constant(value):
        if isinstance(value,types.CodeType):return fingerprint(value)
        if isinstance(value,bytes):return value.hex()
        if isinstance(value,tuple):return [constant(v) for v in value]
        if isinstance(value,frozenset):return sorted(_canonical(constant(v)) for v in value)
        if value is Ellipsis:return 'Ellipsis'
        return _jsonable(value)
    def fingerprint(code):
        return dict(bytecode=code.co_code.hex(),constants=[constant(v) for v in code.co_consts],
            names=code.co_names,variables=code.co_varnames,free=code.co_freevars,
            cells=code.co_cellvars,args=code.co_argcount,kwonly=code.co_kwonlyargcount,
            posonly=code.co_posonlyargcount,flags=code.co_flags & ~0x1000000,
            exceptions=getattr(code,'co_exceptiontable',b'').hex())
    objects=[validate_config,_bwp,_arrays,_raw_cdl_channel,_cdl_spatial_rays,
        _cdl_power_reference,_make_channel,_raw_tdl_channel,_tdl_power_reference,channel_power_reference,antenna_counts,antenna_mode,build_dmrs_layout,linear_weights,
        estimate_pilot_channels,interpolate_channel]
    objects.extend(getattr(DmrsChannelBatch,k) for k in sorted(DmrsChannelBatch.__dict__)
        if callable(getattr(DmrsChannelBatch,k)) and hasattr(getattr(DmrsChannelBatch,k),'__code__'))
    objects.extend(getattr(CdlBatchKernel,k) for k in sorted(CdlBatchKernel.__dict__)
        if callable(getattr(CdlBatchKernel,k)) and hasattr(getattr(CdlBatchKernel,k),'__code__'))
    objects.extend(getattr(TdlBatchKernel,k) for k in sorted(TdlBatchKernel.__dict__)
        if callable(getattr(TdlBatchKernel,k)) and hasattr(getattr(TdlBatchKernel,k),'__code__'))
    native_hash=native_module().implementation_digest()
    return hashlib.sha256(_canonical([[fingerprint(inspect.unwrap(f).__code__) for f in objects],native_hash]).encode()).hexdigest()


def allocation_name(n_slots,interpolate=False,nt=None,nr=None):
    nt=int(N_TX if nt is None else nt);nr=int(N_RX if nr is None else nr)
    prefix='Interp2D ' if interpolate else ''
    return prefix+f'DMRS SC60xSym2 P{nt} nslot{int(n_slots)} Comb2 RB0-9 l2,11'


def slot_label(n_slots):
    return '0' if int(n_slots)==1 else f'0-{int(n_slots)-1}'


def channel_paths(cfg,condition,root):
    prefix='Interp2D ' if cfg.interpolate_2d else ''
    nt,nr=antenna_counts(cfg)
    root=filesystem_path(Path(root)/antenna_mode(nt,nr)/normalized_data_mode(cfg.data_mode))
    allocation=allocation_name(cfg.n_slots,cfg.interpolate_2d,nt,nr)
    if condition['backend'] in ('cost2100','quadriga'):
        config_hash=hashlib.sha256(_canonical(condition['native_settings']).encode()).hexdigest()[:12]
        detail=f"Scene {condition['profile']}, Config {config_hash}, Slot {slot_label(cfg.n_slots)}, Tx{nt} Rx{nr} L{nt}, BWP120"
    else:
        detail=(f"Delay {condition['delay_spread_ns']*1e-9:.9g}, Slot {slot_label(cfg.n_slots)}, "
            f"Doppler {condition['doppler_hz']:g}Hz, Tx{nt} Rx{nr} L{nt}, BWP120")
    return {quantity:filesystem_path(Path(root)/condition_folder(condition)/label/allocation/
        f"{prefix}{label} Channel 2D {condition['snr_db']:g}dB ({detail}).h5")
        for quantity,label in (('ls','LS'),('mmse','MMSE'),('real','Real'))}


def split_ranges(cfg):
    start=0;ranges={}
    for name,count in zip(('train','val','test'),cfg.split_counts):
        ranges[name]=[start,start+int(count)];start+=int(count)
    return ranges


def _create_file(path,quantity,paths,sim,condition,generation_id,implementation):
    cfg=sim.cfg;layout=dict(sim.layout)
    if quantity=='real':
        layout['output_k']=layout['truth_k']
        layout['output_l']=np.arange(layout['rs_l'][0],layout['rs_l'][-1]+1)
        layout['output_rs_mask']=layout['pilot_mask'][np.ix_(layout['output_l'],layout['output_k'])].T
        layout['output_data_mask']=layout['data_mask'][np.ix_(layout['output_l'],layout['output_k'])].T
    nk=len(layout['output_k']);nl=len(layout['output_l'])
    path.parent.mkdir(parents=True,exist_ok=True)
    file=h5py.File(path,'w')
    key={'ls':'LS_Channel','mmse':'MMSE_Channel','real':'Real_Channel'}[quantity]
    file.create_dataset(key,shape=(0,cfg.n_slots,sim.nt,sim.nr,nk,nl),maxshape=(cfg.sample_count,cfg.n_slots,sim.nt,sim.nr,nk,nl),
        chunks=(min(cfg.sample_count,sim.settings.batch_size,32),1,1,1,nk,nl),
        dtype=np.dtype(cfg.compute_dtype),compression=sim.settings.compression)
    for name,value in [('RS_Subcarrier_Indices',layout['rs_k_by_port'][0] if sim.nt==1 else layout['rs_k_by_port']),('RS_Symbol_Indices',layout['rs_l']),
        ('Subcarrier_Indices',layout['output_k']),('Symbol_Indices',layout['output_l']),
        ('RS_Mask',layout['output_rs_mask']),('Data_Mask',layout['output_data_mask']),
        ('Pilot_Allocation_Mask',layout['pilot_mask']),('Data_Allocation_Mask',layout['data_mask'])]:
        file.create_dataset(name,data=value)
    if cfg.source_split_counts is not None:
        file.create_dataset('Source_Sample_Indices',data=sim.source_sample_indices(condition))
    file.create_dataset('RS_Subcarrier_Indices_By_Port',data=layout['rs_k_by_port'])
    file.create_dataset('Port_Indices',data=np.arange(sim.nt))
    file.create_dataset('Data_Subcarrier_Indices',data=layout['data_k'])
    if quantity!='real' and not cfg.interpolate_2d:
        file.attrs['subcarrier_coordinates']='RS_Subcarrier_Indices_By_Port[port,subcarrier_index]'
    file.create_dataset('Slot_Indices',data=np.arange(cfg.n_slots,dtype=np.int32))
    file.create_dataset('Slot_Start_Times_s',data=sim.slot_starts/sim.fs)
    file.create_dataset('Symbol_FFT_Start_Times_s',data=(sim.symbol_starts/sim.fs).reshape(cfg.n_slots,14))
    file.attrs.update(format=FORMAT if cfg.n_slots==1 else MULTI_SLOT_FORMAT,generation_id=generation_id,source_generation_id=generation_id,
        implementation_sha256=implementation,status='partial',num_written=0,target_samples=cfg.sample_count,
        rs_kind='dmrs',quantity=quantity,storage_mode='rs_grid',data_mode=cfg.data_mode,
        axes='sample,slot,port,rx,subcarrier,symbol',index_base=0,slot_index=0,n_slots=cfg.n_slots,slot_stop=cfg.n_slots-1,
        slot_range='Slot '+slot_label(cfg.n_slots),channel_time_continuity=f"one {condition['backend'].upper()} realization per sample; continuous Doppler and FIR history across slots",
        config_json=_canonical(asdict(cfg)),condition_json=_canonical(condition),
        split_ranges_json=_canonical(split_ranges(cfg)),sample_order='train,val,test',
        split_range_convention='zero-based [start,stop)',
        file_set_json=_canonical({'dmrs/'+q:os.path.relpath(v,path.parent) for q,v in paths.items()}),
        interpolate_2d=cfg.interpolate_2d,
        interpolation='none_physical_truth' if quantity=='real' else ('bounded_bilinear' if cfg.interpolate_2d else 'none'),
        cdm_enabled=layout['cdm_enabled'],estimator_policy=cfg.estimator,
        ls_definition='per-RE Y/X for one Tx port; joint FD-CDM LS only for multiple Tx ports; optional bounded bilinear interpolation',
        real_definition='actual physical channel averaged within EACH output RE FFT window; no interpolation across REs',
        mmse_definition='SRS oracle rank-one true-channel covariance per port and RS symbol; same received RS; optional bounded bilinear interpolation',
        lmmse_method='srs_oracle',lmmse_uses_current_real=True,
        noise_variance_re=cfg.tx_epre*10.**(-condition['snr_db']/10) if cfg.noise_enabled else 0.,
        snr_reference='TX EPRE / complex post-FFT AWGN variance; fixed ensemble channel scale',
        channel_power_normalization_json=_canonical(channel_power_reference(cfg,condition)
            if cfg.channel.normalize_average_power else {'mode':'none','amplitude_scale':1.0}),
        slot_duration_s=sim.slot_duration_s,sequence_duration_s=sim.sequence_duration_s,precoder=f'identity_{sim.nt}x{sim.nt}',
        data_symbols_json=_canonical(list(range(3,11))),data_re_count=int(layout['data_mask'].sum()),
        sequence_data_re_count=int(layout['data_mask'].sum())*cfg.n_slots,
        data_modulation=cfg.modulation,csirs_present=False,
        compute_device=str(sim.device),compute_dtype=cfg.compute_dtype,
        neoradium_version=neoradium.__version__,torch_version=str(torch.__version__),
        seed_streams='per sample channel/data/noise; identical draws across SNR and interpolation branches')
    if condition['backend'] in ('cost2100','quadriga'):
        file.attrs['native_source_json']=_canonical(dict(
            backend=condition['backend'],scenario=condition['profile'],pure_python=True,
            matlab_runtime=False,octave_runtime=False,
            implementation_sha256=native_module().implementation_digest(),
            cost_source_commit='bafca0b84faa2aa0441cb3f03f9dbd65771f43b3',
            quadriga_source_commit='277866650eb115adb5b3e8ac252b0d1df073596d',
            scope='one fixed BS/Tx and one MS/Rx track; COST omni / QuaDRiGa legacy and mmMAGIC',
            matlab_numeric_parity_verified=False,
            physical_adapter='time-varying fractional-delay FIR; continuous waveform and FFT-window averaged Real',
            settings=condition['native_settings']))
    file.flush()
    return file


def generate_condition(sim,condition,root):
    validate_condition(condition)
    paths=channel_paths(sim.cfg,condition,root)
    implementation=_implementation_id()
    # Completed compatible data are opened read-only and never recomputed.
    if not sim.settings.overwrite and all(p.is_file() for p in paths.values()):
        complete=True;identities=[]
        for q,path in paths.items():
            with h5py.File(path,'r') as saved:
                identities.append(str(saved.attrs.get('generation_id','')))
                complete &= saved.attrs.get('status')=='complete'
                complete &= int(saved.attrs.get('n_slots',1))==sim.cfg.n_slots
                complete &= int(saved.attrs.get('num_written',0))==sim.cfg.sample_count
                complete &= _compatible_generation_config(json.loads(saved.attrs.get('config_json','{}')),sim.cfg)
                complete &= json.loads(saved.attrs.get('condition_json','{}'))==_jsonable(condition)
                complete &= saved.attrs.get('implementation_sha256') in (*REFERENCE_IMPLEMENTATIONS,implementation)
        if complete and len(set(identities))==1:
            return dict(condition=condition,paths=paths,num_written=sim.cfg.sample_count,generation_id=identities[0])
    generation_id=hashlib.sha256(_canonical([FORMAT if sim.cfg.n_slots==1 else MULTI_SLOT_FORMAT,asdict(sim.cfg),condition,implementation]).encode()).hexdigest()
    keys={'ls':'LS_Channel','mmse':'MMSE_Channel','real':'Real_Channel'}
    exists=[path.exists() for path in paths.values()]
    if any(exists) and not all(exists) and not sim.settings.overwrite:
        raise FileExistsError('Some LS/MMSE/Real files are missing. Preserve the existing set or choose another DATA_ROOT.')
    with ExitStack() as stack:
        files={}
        for quantity,path in paths.items():
            if path.exists() and not sim.settings.overwrite:
                file=stack.enter_context(h5py.File(path,'r+'))
                if file.attrs.get('generation_id')!=generation_id:
                    compatible=(_compatible_generation_config(json.loads(file.attrs.get('config_json','{}')),sim.cfg)
                        and json.loads(file.attrs.get('condition_json','{}'))==_jsonable(condition)
                        and file.attrs.get('implementation_sha256') in (*REFERENCE_IMPLEMENTATIONS,implementation))
                    if not compatible:raise FileExistsError(f'Different generation settings already use: {path}')
                    generation_id=str(file.attrs['generation_id'])
            else:file=stack.enter_context(_create_file(path,quantity,paths,sim,condition,generation_id,implementation))
            files[quantity]=file
        written=[int(f.attrs['num_written']) for f in files.values()]
        start=min(written);target=sim.cfg.sample_count
        for q,file in files.items():
            if not 0<=int(file.attrs['num_written'])<=target or file[keys[q]].shape[0]<int(file.attrs['num_written']):
                raise ValueError('Corrupted committed sample count.')
        if start==target:
            for file in files.values():file.attrs['status']='complete';file.flush()
            return dict(condition=condition,paths=paths,num_written=target,generation_id=generation_id)
        if any(exists) and not sim.settings.resume and not sim.settings.overwrite:
            raise FileExistsError('A partial generation exists; enable RESUME or use another DATA_ROOT.')
        # A batch is committed only after every quantity is flushed. The minimum
        # journal counter recovers a crash between updates without mismatching samples.
        for q,file in files.items():
            file[keys[q]].resize(start,axis=0);file.attrs['num_written']=start;file.attrs['status']='partial';file.flush()
        progress=tqdm(total=target,initial=start,disable=not sim.settings.show_progress,
            desc=condition_text(condition), dynamic_ncols=True, unit='sample')
        try:
            for lo in range(start,target,sim.effective_batch_size):
                hi=min(lo+sim.effective_batch_size,target)
                batch=sim.simulate(condition,range(lo,hi))
                for q,file in files.items():
                    file[keys[q]].resize(hi,axis=0);file[keys[q]][lo:hi]=batch[q];file.flush()
                for file in files.values():file.attrs['num_written']=hi;file.flush()
                progress.update(hi-lo)
            for file in files.values():file.attrs['status']='complete';file.flush()
        finally:progress.close()
    return dict(condition=condition,paths=paths,num_written=target,generation_id=generation_id)


def generate_all_conditions(cfg,settings,conditions,root):
    simulator=DmrsChannelBatch(cfg,settings);outputs=[]
    print('연속 슬롯:',cfg.n_slots,'| 슬롯 범위: Slot '+slot_label(cfg.n_slots),
          '| 실제 생성 배치:',simulator.effective_batch_size,'| 기준 배치:',settings.batch_size)
    for condition in conditions:
        output=generate_condition(simulator,condition,root)
        output['split_counts']=list(cfg.split_counts)
        outputs.append(output)
    return outputs


def read_channel_sample(ls_path,sample=0,slot=0,port=0,rx=0):
    ls_path=filesystem_path(ls_path)
    with h5py.File(ls_path,'r') as file:
        if file.attrs.get('format') not in (FORMAT,MULTI_SLOT_FORMAT,RANDOM_MIX_FORMAT,MULTI_SLOT_MIX_FORMAT,'dmrs-random-test-1.0','dmrs-multi-slot-random-test-1.0'):raise ValueError('Not a DM-RS channel dataset.')
        if sample<0 or sample>=int(file.attrs['num_written']):raise IndexError('Sample is not committed.')
        n_slots=int(file.attrs.get('n_slots',1))
        if not isinstance(slot,(int,np.integer)) or not 0<=slot<n_slots:raise IndexError('Requested slot is outside saved slots.')
        paths={q:filesystem_path(ls_path.parent/v) for q,v in json.loads(file.attrs['file_set_json']).items()}
        identity=str(file.attrs['generation_id'])
        result={name:file[name][...] for name in ('Subcarrier_Indices','Symbol_Indices','RS_Mask','Data_Mask')}
        result.update(slot_index=int(slot),n_slots=n_slots)
    for quantity,key in (('ls','LS_Channel'),('mmse','MMSE_Channel'),('real','Real_Channel')):
        with h5py.File(paths['dmrs/'+quantity],'r') as file:
            if file.attrs['generation_id']!=identity or sample>=int(file.attrs['num_written']):
                raise ValueError('The LS/MMSE/Real file set is mismatched or incomplete.')
            if file[key].shape[1]!=n_slots:raise ValueError('LS/MMSE/Real slot counts differ.')
            result[quantity]=file[key][sample,slot,port,rx]
            result[quantity+'_k']=(file['RS_Subcarrier_Indices_By_Port'][port] if quantity!='real' and not file.attrs['interpolate_2d'] else file['Subcarrier_Indices'][...])
            result[quantity+'_l']=file['Symbol_Indices'][...]
    return result


"""Embedded helpers shared by the generator and four model notebooks."""









def filesystem_path(path):
    value=str(path)
    if value.startswith('\\\\?\\UNC\\'):value='\\\\'+value[8:]
    elif value.startswith('\\\\?\\'):value=value[4:]
    path=Path(os.path.abspath(os.path.normpath(value)))
    if os.name!='nt':return path
    value=str(path)
    return Path('\\\\?\\UNC\\'+value[2:] if value.startswith('\\\\') else '\\\\?\\'+value)

def balanced_split_quotas(environment_count,counts,seed):
    if environment_count<1:raise ValueError('Mix 환경 리스트가 비어 있습니다.')
    result=[]
    for part,count in enumerate(counts):
        rng=np.random.default_rng(np.random.SeedSequence([int(seed),part]))
        quota=np.full(environment_count,int(count)//environment_count,dtype=np.int64)
        quota[rng.permutation(environment_count)[:int(count)%environment_count]]+=1
        result.append(quota)
    return np.asarray(result)

"""Embedded Random-channel HDF5 layout used by generation and model loading."""

RANDOM_MIX_FORMAT = 'dmrs-random-mix-1.0'
MULTI_SLOT_MIX_FORMAT = 'dmrs-multi-slot-random-mix-1.0'


def random_mix_paths(root,channel_type,interpolate=False,n_slots=None,nt=None,nr=None,purpose='train_val'):
    n_slots=int(N_SLOTS if n_slots is None else n_slots)
    if purpose not in ('train_val','test'):raise ValueError('Random purpose must be train_val or test.')
    kind='Random Test' if purpose=='test' else 'Random'
    prefix='Interp2D ' if interpolate else ''
    allocation=allocation_name(n_slots,interpolate,nt,nr)
    plain=allocation_name(n_slots,False,nt,nr)
    nt=int(N_TX if nt is None else nt);nr=int(N_RX if nr is None else nr)
    root=filesystem_path(Path(root)/antenna_mode(nt,nr)/'Mix')
    return {quantity:filesystem_path(Path(root)/channel_type/f'{kind} {prefix}{label} Channel'/allocation/
            f'{kind} {prefix}{label} Channel 2D ({plain}).h5')
            for quantity,label in [('ls','LS'),('mmse','MMSE'),('real','Real')]}


def random_mix_config_path(paths):
    return paths['ls'].parent/('Random Test Config.txt' if 'Random Test' in paths['ls'].name else 'Random Channel Config.txt')


def read_random_mix(paths):
    metadata=None;identity=None;sources=None;first_order=None
    keys={'ls':'LS_Channel','mmse':'MMSE_Channel','real':'Real_Channel'}
    for q,path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(f'Random {q.upper()} 파일이 없습니다: {path}\nData Generation의 마지막 Mix 셀을 실행하세요.')
        with h5py.File(path,'r') as file:
            if file.attrs.get('format') not in (RANDOM_MIX_FORMAT,MULTI_SLOT_MIX_FORMAT,'dmrs-random-test-1.0','dmrs-multi-slot-random-test-1.0') or file.attrs.get('status')!='complete':
                raise ValueError(f'Random 채널 저장이 완료되지 않았습니다: {path}')
            if json.loads(file.attrs['config_json'])['carrier']['scs_khz']!=15:
                raise ValueError('이 실험은 15 kHz Random 채널을 사용합니다.')
            current=json.loads(file['Metadata_JSON'].asstr()[()])
            slots=int(current.get('n_slots',1))
            if file[keys[q]].shape[1]!=slots or int(file.attrs.get('n_slots',1))!=slots:
                raise ValueError('Random 채널의 실제 슬롯 수가 설정과 다릅니다.')
            current_id=str(file.attrs['generation_id'])
            current_sources=json.loads(file['Sources_JSON'].asstr()[()])
            order=tuple(file[k][...] for k in ('Source_Index','Source_Row','Split_Index'))
            if metadata is None:metadata=current;identity=current_id;sources=current_sources;first_order=order
            elif (metadata!=current or identity!=current_id or sources!=current_sources or
                  any(not np.array_equal(a,b) for a,b in zip(first_order,order))):
                raise ValueError('Random LS/MMSE/Real의 환경·분할·샘플 순서가 서로 다릅니다.')
            count=sum(current['counts'])
            if file[keys[q]].shape[0]!=count or int(file.attrs.get('num_written',0))!=count:
                raise ValueError('Random 채널의 저장 샘플 수가 설정과 다릅니다.')
            bounds=np.cumsum([0]+current['counts'])
            for part in range(3):
                if not np.all(order[2][bounds[part]:bounds[part+1]]==part):
                    raise ValueError('Random Train / Val / Test의 저장 순서가 다릅니다.')
    expected=hashlib.sha256(_canonical(metadata).encode()).hexdigest()
    if identity!=expected:raise ValueError('Random 채널 설정과 generation_id가 다릅니다.')
    return dict(paths={q:str(p) for q,p in paths.items()},path=str(paths['ls']),mix_id=identity,
                metadata=metadata,sources=sources)


def generate_test_conditions(cfg, settings, conditions, root):
    """환경별 Test만 생성. 완료된 기존 원본은 해당 Test 구간을 그대로 재사용합니다."""
    test_cfg = copy.deepcopy(cfg)
    test_cfg.split_counts = (0, 0, int(cfg.split_counts[2]))
    test_cfg.source_split_counts = tuple(cfg.split_counts)
    simulator = None
    outputs = []
    implementation = _implementation_id()
    for condition in conditions:
        paths = channel_paths(test_cfg, condition, root)
        reuse = None
        if not settings.overwrite and all(p.is_file() for p in paths.values()):
            identities = []
            for q, path in paths.items():
                with h5py.File(path, 'r') as saved:
                    saved_cfg = json.loads(saved.attrs.get('config_json', '{}'))
                    expected = _jsonable(asdict(cfg))
                    actual = copy.deepcopy(saved_cfg)
                    actual.setdefault('n_slots', 1)
                    for key in ('split_counts', 'source_split_counts', 'source_selection_seed'):
                        expected.pop(key, None)
                        actual.pop(key, None)
                    ranges = json.loads(saved.attrs.get('split_ranges_json', '{}'))
                    lo, hi = ranges.get('test', [0, 0])
                    key = {'ls': 'LS_Channel', 'mmse': 'MMSE_Channel', 'real': 'Real_Channel'}[q]
                    compatible = (saved.attrs.get('format') in (FORMAT, MULTI_SLOT_FORMAT)
                        and saved.attrs.get('status') == 'complete' and expected == actual
                        and saved.attrs.get('implementation_sha256') in (*REFERENCE_IMPLEMENTATIONS,implementation)
                        and json.loads(saved.attrs.get('condition_json', '{}')) == _jsonable(condition)
                        and hi - lo == test_cfg.sample_count
                        and int(saved.attrs.get('num_written', -1)) == hi
                        and saved[key].shape[0] == hi and saved[key].shape[1] == cfg.n_slots)
                    # Test-only files retain the original sample coordinates for link replay.
                    if compatible:
                        indices = np.arange(lo, hi, dtype=np.int64)
                        if 'Source_Sample_Indices' in saved:
                            indices = saved['Source_Sample_Indices'][lo:hi]
                        expected_indices = np.arange(sum(cfg.split_counts[:2]), sum(cfg.split_counts))
                        compatible = np.array_equal(indices, expected_indices)
                    if not compatible:
                        break
                    identities.append(str(saved.attrs['generation_id']))
                    reuse = dict(condition=condition, paths=paths,
                        num_written=hi, split_counts=saved_cfg['split_counts'],
                        generation_id=identities[-1], test_samples=hi-lo, reused=True)
            if len(identities) != 3 or len(set(identities)) != 1:
                reuse = None
        if reuse is not None:
            print('기존 Test 재사용:',condition_text(condition))
            outputs.append(reuse)
            continue
        if simulator is None:
            simulator = DmrsChannelBatch(test_cfg, settings)
            print('환경별 Test 생성 | 실제 배치:', simulator.effective_batch_size,
                  '| 환경당 Test:', test_cfg.sample_count, '| nslot:', cfg.n_slots)
        output = generate_condition(simulator, condition, root)
        output.update(split_counts=list(test_cfg.split_counts),
                      test_samples=test_cfg.sample_count, reused=False)
        outputs.append(output)
    return outputs


def direct_mix_plan(cfg, conditions, counts, seed):
    """환경별 필요 수량과 물리 샘플 번호를 정한 뒤 split별 마지막 셔플을 고정합니다."""
    if len(counts)!=3 or any(isinstance(n,(bool,np.bool_)) or not isinstance(n,(int,np.integer)) or n<0 for n in counts):
        raise ValueError('Random split counts must be three nonnegative integers.')
    train_val=counts[0]>0 and counts[1]>0 and counts[2]==0
    test=counts[0]==0 and counts[1]==0 and counts[2]>0
    if not (train_val or test):raise ValueError('Use Train/Val-only or Test-only Random data.')
    if test and counts[2]<len(conditions):raise ValueError('Random Test 총수는 환경 조합 수 이상이어야 각 환경에 한 장 이상 배분됩니다.')
    quotas = balanced_split_quotas(len(conditions), counts, seed)
    rng = np.random.default_rng(seed)
    source_ids, source_rows, parts, permutations = [], [], [], []
    ranges = split_ranges(cfg)
    start = 0
    for part, (name, count) in enumerate(zip(('train', 'val', 'test'), counts)):
        groups, rows = [], []
        lo, hi = ranges[name]
        for index in range(len(conditions)):
            take = int(quotas[part, index])
            if take > hi - lo:
                raise ValueError(f'{name}: 환경별 생성 수량이 샘플 번호 구간보다 큽니다.')
            groups.append(np.full(take, index, dtype=np.int32))
            rows.append(rng.choice(hi-lo, take, replace=False).astype(np.int64) + lo)
        source_ids.append(np.concatenate(groups))
        source_rows.append(np.concatenate(rows))
        parts.append(np.full(count, part, dtype=np.int8))
        permutations.append(rng.permutation(count) + start)
        start += count
    return (np.concatenate(source_ids), np.concatenate(source_rows), np.concatenate(parts),
            np.concatenate(permutations), quotas)


def write_direct_mix_config(paths, metadata):
    is_test=metadata.get('purpose')=='test'
    text = [f'Channels: {metadata["channel_types"]}',
        f'Estimator: {metadata["config"]["estimator"]}',
        f'Antennas: {metadata["config"]["antennas"]}',
        f'Slots: {metadata["n_slots"]} (Slot {slot_label(metadata["n_slots"])})',
        'Generation: direct balanced Random Test; total Test shared by all environments.' if is_test else 'Generation: direct balanced Train/Val; no per-environment Train/Val files.',
        f'SNR (dB): {metadata["snr_list"]}', f'Delay (ns): {metadata["delay_list_ns"]}',
        f'Doppler (Hz): {metadata["doppler_list_hz"]}',
        f'Environment combinations N: {metadata["environment_count"]}',
        f'Train / Val / Test: {metadata["counts"]}', f'Mix seed: {metadata["seed"]}',
        f'Data generation seed: {metadata["data_seed"]}',
        'Each environment generates its 1/N quota directly; remainders differ by at most one.',
        'Test shuffled once after all environments are generated.' if is_test else 'Train and Val are independently shuffled after generation.',
        'LS/MMSE/Real share identical Source_Index, Source_Row and Split_Index.',
        'Source_Row is a physical sample seed coordinate, not an external HDF5 row dependency.',
        'One row contains a complete continuous-slot trajectory; slots are never shuffled separately.',
        'This file is Test-only; Split_Index=2; physical rows come only from the Test seed range.' if is_test else 'Test is in separate Random Test files; this file contains Train/Val only.',
        'Real is the physical channel, without interpolation.']
    text+=['Native scenarios (Delay/Doppler come from geometry, not TDL/CDL lists):']
    text += [_canonical(c) for c in metadata['conditions'] if c['backend'] in ('cost2100','quadriga')]
    random_mix_config_path(paths).write_text('\n'.join(text)+'\n', encoding='utf-8')


def generate_random_mix(cfg, settings, root, channel_types, delays, dopplers, snrs, counts, seed, purpose='train_val'):
    """환경별 quota를 직접 생성하고 동일 Pair 순열로 Train/Val 또는 Test Random 파일 저장."""
    if purpose not in ('train_val','test'):raise ValueError('Random purpose must be train_val or test.')
    if purpose=='test' and tuple(counts)!=(0,0,int(cfg.split_counts[2])):raise ValueError('Random Test counts must be (0,0,TEST_SAMPLES).')
    validate_config(cfg)
    if cfg.source_split_counts is not None:
        raise ValueError('직접 Mix의 cfg.source_split_counts는 None이어야 합니다.')
    if cfg.data_mode!='Mix':raise ValueError('직접 Mix 생성은 DATA_MODE=Mix에서만 사용합니다.')
    channel_types=validate_channel_selection([channel_types] if isinstance(channel_types,str) else list(channel_types))
    channel_type=channel_types[0] if len(channel_types)==1 else 'Mixed'
    conditions = make_conditions(channel_types, delays, dopplers, snrs)
    conditions.sort(key=condition_sort_key)
    source_ids, source_rows, parts, permutation, quotas = direct_mix_plan(cfg, conditions, counts, seed)
    implementation = _implementation_id()
    sources = [dict(condition=c, config=_jsonable(asdict(cfg)), split_ranges=split_ranges(cfg),
        files={}, storage='direct_generated',
        generation_id=hashlib.sha256(_canonical([asdict(cfg), c, implementation]).encode()).hexdigest(),
        implementation_sha256=implementation) for c in conditions]
    metadata = dict(format=RANDOM_MIX_FORMAT if cfg.n_slots == 1 else MULTI_SLOT_MIX_FORMAT,
        n_slots=cfg.n_slots, slot_indices=list(range(cfg.n_slots)), slot_range='Slot '+slot_label(cfg.n_slots),
        name=channel_type, channel_type=channel_type, data_mode=cfg.data_mode, seed=int(seed), data_seed=int(cfg.seed),
        config=_jsonable(asdict(cfg)),implementation_sha256=implementation,
        counts=list(map(int, counts)), channel_types=channel_types,
        snr_list=sorted(map(float, snrs)), delay_list_ns=sorted(map(float, delays)),
        doppler_list_hz=sorted(map(float, dopplers)), environment_count=len(conditions),
        environment_quotas=quotas.T.tolist(), conditions=conditions,
        source_generation_ids=[s['generation_id'] for s in sources],
        split_ranges=[s['split_ranges'] for s in sources],
        generation_mode='direct-balanced-train-val-v1', source_files_required=False,
        sampling='generate only 1/N Train/Val per condition; final independent split shuffle; no Mix Test')
    if purpose=='test':
        metadata.update(format='dmrs-random-test-1.0' if cfg.n_slots==1 else 'dmrs-multi-slot-random-test-1.0',
            purpose='test',generation_mode='direct-balanced-test-v1',
            sampling='total Test count shared across all Test-list environments; one paired final shuffle; no Train/Val rows')
    identity = hashlib.sha256(_canonical(metadata).encode()).hexdigest()
    paths = random_mix_paths(root, channel_type, cfg.interpolate_2d, cfg.n_slots,*antenna_counts(cfg),purpose=purpose)
    # Keep completed TDL/CDL Mix files from the exact previous implementation,
    # whose numerical channels are checked bitwise against this source snapshot.
    if not settings.overwrite and all(path.is_file() for path in paths.values()):
        legacy=[]
        for path in paths.values():
            with h5py.File(path,'r') as saved:
                prior=json.loads(saved['Metadata_JSON'][()].decode('utf-8'))
                left=copy.deepcopy(prior);right=copy.deepcopy(metadata)
                for value in (left,right):
                    value.pop('implementation_sha256',None);value.pop('source_generation_ids',None)
                accepted=(saved.attrs.get('status')=='complete' and left==right
                    and prior.get('implementation_sha256') in REFERENCE_IMPLEMENTATIONS
                    and all(c['backend'] in ('tdl','cdl') for c in prior['conditions']))
                legacy.append(accepted)
        if all(legacy):
            result=read_random_mix(paths)
            write_direct_mix_config(paths,result['metadata'])
            print('기존 호환 TDL/CDL Mix 재사용:',paths['ls'])
            return result
    existing = []
    for q, path in paths.items():
        if not path.is_file():
            continue
        with h5py.File(path, 'r') as saved:
            same = saved.attrs.get('generation_id') == identity and saved.attrs.get('status') == 'complete'
        if not same and not settings.overwrite:
            raise FileExistsError(f'다른 설정의 Random 파일이 존재합니다: {path}\n'
                '기존 파일은 변경하지 않았습니다. 별도 DATA_ROOT를 사용하거나 OVERWRITE를 명시하세요.')
        if same:
            existing.append(q)
    if len(existing) == 3 and not settings.overwrite:
        result = read_random_mix(paths)
        write_direct_mix_config(paths, metadata)
        for path in paths.values():
            for suffix in ('work', 'shuffle'):
                temporary = path.parent/f'.Random_DirectMix_{identity[:20]}.{suffix}.h5'
                if temporary.exists():
                    temporary.unlink()
        print('기존 직접 생성 Mix 로드:', paths['ls'])
        return result
    raw_paths = {q: p.parent/f'.Random_DirectMix_{identity[:20]}.work.h5' for q, p in paths.items()}
    shuffled_paths = {q: p.parent/f'.Random_DirectMix_{identity[:20]}.shuffle.h5' for q, p in paths.items()}
    if any(p.exists() for p in raw_paths.values()) and not settings.resume and not settings.overwrite:
        raise FileExistsError('직접 Mix 생성 중간 파일이 있습니다. RESUME=True로 재개하세요.')
    if existing and not settings.overwrite and not all(p.exists() for p in raw_paths.values()):
        raise FileExistsError('Random 파일 일부만 존재하고 재개할 중간 파일이 없습니다.')
    mixed_cfg = copy.deepcopy(cfg)
    mixed_cfg.split_counts = tuple(map(int, counts))
    count = sum(counts)
    keys = {'ls': 'LS_Channel', 'mmse': 'MMSE_Channel', 'real': 'Real_Channel'}
    sim = DmrsChannelBatch(mixed_cfg, settings)
    print(('Random Test' if purpose=='test' else '직접 Mix')+' | 환경:', len(conditions), '| Train / Val / Test:', list(counts),
          '| 실제 생성 배치:', sim.effective_batch_size, '| nslot:', cfg.n_slots)
    # Only three shared working files are used; no environment-level Train/Val file is written.
    # Journals commit after all three quantities are flushed, so a crash can safely replay a batch.
    with ExitStack() as stack:
        files = {}
        for q, path in raw_paths.items():
            if path.exists() and not settings.overwrite:
                target = stack.enter_context(h5py.File(path, 'r+'))
                if target.attrs.get('generation_id') != identity or target[keys[q]].shape[0] != count:
                    raise ValueError(f'직접 Mix 중간 파일의 설정이 다릅니다: {path}')
            else:
                target = stack.enter_context(_create_file(path, q, paths, sim, conditions[0], identity, implementation))
                target[keys[q]].resize(count, axis=0)
                for name in ('condition_json', 'noise_variance_re', 'channel_power_normalization_json'):
                    if name in target.attrs:
                        del target.attrs[name]
                for name, value in [('Source_Index', source_ids), ('Source_Row', source_rows), ('Split_Index', parts)]:
                    target.create_dataset(name, data=value)
                target.create_dataset('Metadata_JSON', data=_canonical(metadata), dtype=h5py.string_dtype('utf-8'))
                target.create_dataset('Sources_JSON', data=_canonical(sources), dtype=h5py.string_dtype('utf-8'))
                target.attrs.update(format=metadata['format'], mix_id=identity,
                    storage_mode='random_mix', generation_mode=metadata['generation_mode'],
                    phase='generating', num_generated=0, num_written=0, target_samples=count,
                    physical_source_split_counts_json=_canonical(cfg.split_counts),
                    source_row_definition='physical sample seed coordinate; no source file is required')
                target.flush()
            files[q] = target
        committed = min(int(f.attrs['num_generated']) for f in files.values())
        if not 0 <= committed <= count:
            raise ValueError('직접 Mix 생성 진행 정보가 손상되었습니다.')
        for f in files.values():
            f.attrs.update(num_generated=committed, phase='generating')
            f.flush()
        with tqdm(total=count, initial=committed, desc='Random Test generation' if purpose=='test' else 'Direct Mix Train/Val', unit='sample',
                  dynamic_ncols=True, disable=not settings.show_progress) as bar:
            visited = 0
            for index, condition in enumerate(conditions):
                for part in (0, 1, 2):
                    positions = np.flatnonzero((source_ids == index) & (parts == part))
                    done = min(len(positions), max(0, committed-visited))
                    for lo in range(done, len(positions), sim.effective_batch_size):
                        dest = positions[lo:lo+sim.effective_batch_size]
                        values = sim.simulate(condition, source_rows[dest])
                        # Each environment/split occupies one contiguous interval before shuffling.
                        begin, end = int(dest[0]), int(dest[-1])+1
                        for q, f in files.items():
                            f[keys[q]][begin:end] = values[q]
                            f.flush()
                        committed = visited+lo+len(dest)
                        for f in files.values():
                            f.attrs['num_generated'] = committed
                            f.flush()
                        bar.set_postfix_str(condition_text(condition)+' / '+('Train','Val','Test')[part],refresh=False)
                        bar.update(len(dest))
                    visited += len(positions)
        assert committed == count
        for f in files.values():
            f.attrs['phase'] = 'generated'
            f.flush()
    # Gather shuffled source rows into contiguous output blocks; never load all channel rows into RAM.
    for q in keys:
        with h5py.File(raw_paths[q], 'r') as source, h5py.File(shuffled_paths[q], 'w') as target:
            for name, value in source.attrs.items():
                target.attrs[name] = value
            for name in source:
                if name not in (keys[q], 'Source_Index', 'Source_Row', 'Split_Index'):
                    source.copy(name, target)
            data = source[keys[q]]
            output = target.create_dataset(keys[q], shape=data.shape, dtype=data.dtype,
                chunks=data.chunks, compression=settings.compression)
            for name, values in [('Source_Index', source_ids), ('Source_Row', source_rows), ('Split_Index', parts)]:
                target.create_dataset(name, data=values[permutation])
            target.attrs.update(status='partial', phase='shuffling', num_written=0)
            row_bytes = int(np.prod(data.shape[1:]))*data.dtype.itemsize
            batch = min(settings.batch_size, max(1, (64*1024**2)//max(1, row_bytes)))
            with tqdm(total=count, desc=f'Final shuffle {q.upper()}', unit='sample',
                      dynamic_ncols=True, disable=not settings.show_progress) as bar:
                for lo in range(0, count, batch):
                    selected = permutation[lo:lo+batch]
                    order = np.argsort(selected)
                    output[lo:lo+len(selected)] = data[selected[order]][np.argsort(order)]
                    target.attrs['num_written'] = lo+len(selected)
                    bar.update(len(selected))
            target.attrs.update(status='complete', phase='complete', num_written=count)
            target.flush()
    # All HDF5 handles are closed before publication (also required on Windows).
    for q, path in paths.items():
        os.replace(shuffled_paths[q], path)
    result = read_random_mix(paths)
    write_direct_mix_config(paths, metadata)
    for path in raw_paths.values():
        path.unlink()
    print(('Random Test' if purpose=='test' else '직접 Mix')+' 저장 완료:', count, 'samples |', len(conditions), 'environments')
    for q, path in paths.items():
        print(q, ':', path)
    return result


## RS / 데이터 영역 확인과 채널 Plot


In [ ]:
def plot_allocation(cfg):
    from matplotlib.colors import ListedColormap,BoundaryNorm
    layout=build_dmrs_layout(cfg)
    image=np.zeros((120,14),dtype=int)
    image[layout['data_mask'].T]=1;image[layout['pilot_mask'].T]=2
    fig,ax=plt.subplots(figsize=(10,5))
    plot=ax.imshow(image,origin='lower',aspect='auto',interpolation='nearest',
        cmap=ListedColormap(['#f1f3f5','#90caf9','#d84315']),norm=BoundaryNorm([-.5,.5,1.5,2.5],3))
    ax.set(xlabel='OFDM symbol index',ylabel='Subcarrier index',xticks=range(14),
        title=f"Tx{layout['nt']} Rx{layout['nr']} / nslot{cfg.n_slots} / CDM {'On' if layout['cdm_enabled'] else 'Off'} / DM-RS 60 x 2 per port")
    bar=fig.colorbar(plot,ax=ax,ticks=[0,1,2]);bar.ax.set_yticklabels(['Unused','Random QAM data','DM-RS'])
    fig.tight_layout();return fig


def plot_channels(channels):
    magnitude_max=max(float(np.abs(channels[q]).max()) for q in ('ls','mmse','real'))
    fig,axes=plt.subplots(2,3,figsize=(14,7),sharex=True,sharey=True)
    for column,(q,label) in enumerate((('ls','LS'),('mmse','LMMSE (oracle)'),('real','Real (physical H)'))):
        value=channels[q];k=channels[q+'_k'];ell=channels[q+'_l']
        for row,(values,title,cmap,limits) in enumerate(((np.abs(value),'Magnitude','viridis',(0,magnitude_max)),
            (np.angle(value),'Phase','twilight',(-np.pi,np.pi)))):
            ax=axes[row,column]
            plot=ax.pcolormesh(ell,k,values,shading='nearest',cmap=cmap,vmin=limits[0],vmax=limits[1])
            ax.set(title=f"Slot {channels.get('slot_index',0)} / "+label+' — '+title,xlabel='OFDM symbol index',ylabel='Subcarrier index')
            fig.colorbar(plot,ax=ax)
    fig.tight_layout();return fig


def run_geometry_checks(cfg):
    layout=build_dmrs_layout(cfg)
    assert layout['pilot_masks'].sum()==layout['nt']*120
    assert not np.any(layout['pilot_mask'] & layout['data_mask'])
    assert np.array_equal(layout['pilot_masks'][:,2],layout['pilot_masks'][:,11])
    assert layout['data_mask'].sum()==len(layout['data_k'])*8
    for p,k in enumerate(layout['rs_k_by_port']):
        assert not np.array_equal(layout['pilot_grid'][p,2,k],layout['pilot_grid'][p,11,k])
    assert layout['cdm_enabled']==(layout['nt']>1)
    assert layout['inverse'].shape[-1]==(1 if layout['nt']==1 else 2)
    k=layout['rs_k_by_port'][:,:,None];ell=layout['rs_l'][None,None,:]
    values=(1+.2*k+.3*ell+.01*k*ell)+1j*(2-.1*k+.4*ell-.02*k*ell)
    actual=interpolate_channel(torch.as_tensor(values[None,:,None],dtype=torch.complex128),layout)[0,:,0].numpy()
    q=(np.broadcast_to(layout['output_k'],(layout['nt'],len(layout['output_k'])))
       if cfg.interpolate_2d else layout['rs_k_by_port'])[:,:,None]
    t=layout['output_l'][None,None,:]
    expected=(1+.2*q+.3*t+.01*q*t)+1j*(2-.1*q+.4*t-.02*q*t)
    np.testing.assert_allclose(actual,expected,rtol=1e-13,atol=1e-13)
    print('RS positions/sequences, automatic CDM and bounded interpolation: passed')


# 설정 검사는 파일 생성 전에 끝냅니다. 지원하지 않는 조합/생성기는 여기서 알립니다.
CONFIG,EXECUTION=make_notebook_settings()
SELECTED_CHANNEL_TYPES=validate_channel_selection(CHANNEL_TYPE_LIST)
CONDITIONS=make_conditions(SELECTED_CHANNEL_TYPES,DELAY_SPREAD_LIST_NS,DOPPLER_LIST_HZ,SNR_LIST_DB)
MIX_CONDITIONS=(make_conditions(SELECTED_CHANNEL_TYPES,MIX_DELAY_LIST_NS,MIX_DOPPLER_LIST_HZ,MIX_SNR_LIST)
                if CONFIG.data_mode=='Mix' else [])
print('DATA_MODE:',CONFIG.data_mode,'| 채널/프로파일:',SELECTED_CHANNEL_TYPES)
print('환경 조합:',len(CONDITIONS),'| slot:',CONFIG.n_slots)
for i,c in enumerate(CONDITIONS):
    print(f'[{i:3}]',condition_text(c))
run_geometry_checks(CONFIG)
PREFLIGHT_LAYOUT=build_dmrs_layout(CONFIG)
nt,nr=antenna_counts(CONFIG)
rs_values=60*2 if not INTERP else len(PREFLIGHT_LAYOUT['output_k'])*len(PREFLIGHT_LAYOUT['output_l'])
real_values=len(PREFLIGHT_LAYOUT['truth_k'])*len(PREFLIGHT_LAYOUT['truth_l'])
bytes_per_sample=CONFIG.n_slots*nt*nr*(2*rs_values+real_values)*np.dtype(CONFIG.compute_dtype).itemsize
if CONFIG.data_mode=='Fix':
    total=len(CONDITIONS)*CONFIG.sample_count
    print('환경별 Train/Val/Test:',CONFIG.split_counts,'| 전체:',total,'samples')
else:
    total=TEST_SAMPLES+TRAIN_SAMPLES+VAL_SAMPLES
    print('Random Test 총수:',TEST_SAMPLES,'| Test 환경:',len(CONDITIONS),'| Train/Val 환경:',len(MIX_CONDITIONS),'| Mix 전체 Train/Val:',TRAIN_SAMPLES,VAL_SAMPLES)
print('LS/MMSE/Real 전체 예상 크기 (압축 전):',round(bytes_per_sample*total/1e9,3),'GB')
print('저장 위치:',DATA_ROOT/antenna_mode(nt,nr)/CONFIG.data_mode)
ALLOCATION_FIGURE=plot_allocation(CONFIG)
plt.show()


## Fix 전체 split / Mix Random Test 생성

Fix는 환경마다 Train/Val/Test를 생성합니다. Mix는 Test 목록 전체에 Test 총수를 균등 배분한 별도 Random Test 세트를 생성합니다.


In [ ]:
CONFIG,EXECUTION=make_notebook_settings()
SELECTED_CHANNEL_TYPES=validate_channel_selection(CHANNEL_TYPE_LIST)
CONDITIONS=make_conditions(SELECTED_CHANNEL_TYPES,DELAY_SPREAD_LIST_NS,DOPPLER_LIST_HZ,SNR_LIST_DB)
OUTPUTS=[];RANDOM_TEST_CHANNELS=None
if CONFIG.data_mode=='Fix':
    print('Fix: 환경마다 Train/Val/Test',CONFIG.split_counts,'생성')
    OUTPUTS=generate_all_conditions(CONFIG,EXECUTION,CONDITIONS,DATA_ROOT)
else:
    print('Mix: Test 전체',TEST_SAMPLES,'개를',len(CONDITIONS),'개 환경에 배분합니다.')
    RANDOM_TEST_CHANNELS=generate_random_mix(CONFIG,EXECUTION,DATA_ROOT,CHANNEL_TYPE_LIST,
        DELAY_SPREAD_LIST_NS,DOPPLER_LIST_HZ,SNR_LIST_DB,(0,0,TEST_SAMPLES),MIX_SEED,purpose='test')
    OUTPUTS=[dict(paths={q:filesystem_path(p) for q,p in RANDOM_TEST_CHANNELS['paths'].items()})]
expected=CONFIG.split_counts if CONFIG.data_mode=='Fix' else (0,0,TEST_SAMPLES)
for output in OUTPUTS:
    for quantity,path in output['paths'].items():
        with h5py.File(path,'r') as saved:
            ranges=json.loads(saved.attrs['split_ranges_json'])
            actual=tuple(ranges[p][1]-ranges[p][0] for p in ('train','val','test'))
            if saved.attrs.get('status')!='complete' or actual!=tuple(expected):
                raise RuntimeError('모드의 Train/Val/Test 수량과 저장 파일이 다릅니다: '+str(path))
        print(quantity,':',path)
print(CONFIG.data_mode,'저장 완료')


## 환경별 Test 저장 결과 확인

In [ ]:
PLOT_SAMPLE=0
PLOT_SLOT=0  # 저장된 슬롯 중 확인할 슬롯 번호: 0 … N_SLOTS-1
SAVED_LS_FILE=None  # 이미 생성한 파일을 확인하려면 LS .h5 경로를 지정
if SAVED_LS_FILE is not None or globals().get('OUTPUTS',[]):
    path=Path(SAVED_LS_FILE) if SAVED_LS_FILE is not None else OUTPUTS[-1]['paths']['ls']
    SAVED_CHANNELS=read_channel_sample(path,PLOT_SAMPLE,PLOT_SLOT)
    print('채널 크기:',SAVED_CHANNELS['real'].shape,
          '| 실제 subcarrier:',SAVED_CHANNELS['real_k'][[0,-1]],
          '| 실제 symbol:',SAVED_CHANNELS['real_l'])
    CHANNEL_FIGURE=plot_channels(SAVED_CHANNELS)
    plt.show()


## Mix 전체 Train·Val 생성 — Fix에서는 건너뜀

MIX 목록에서 Train 총 10만·Val 총 3만을 균등 배분합니다. 위 Random Test와 별도 파일이며, LS/MMSE/Real을 같은 순열로 섞습니다.


In [ ]:
# 채널 종류/프로파일은 상단 CHANNEL_TYPE_LIST를 공통으로 사용합니다.
# Mix 환경별 Delay/Doppler/SNR도 상단 MIX_*_LIST에서 지정합니다.
MIX_CONFIG,MIX_EXECUTION=make_notebook_settings()
RANDOM_CHANNELS=None
if MIX_CONFIG.data_mode=='Mix':
    RANDOM_CHANNELS=generate_random_mix(
        MIX_CONFIG,MIX_EXECUTION,DATA_ROOT,CHANNEL_TYPE_LIST,
        MIX_DELAY_LIST_NS,MIX_DOPPLER_LIST_HZ,MIX_SNR_LIST,
        (TRAIN_SAMPLES,VAL_SAMPLES,0),MIX_SEED)
else:
    print('Fix 모드: 위 환경별 생성 셀에서 Train/Val/Test를 모두 저장하므로 별도 Mix를 만들지 않습니다.')
